In [1]:
from google.colab import drive
from google.colab import userdata

drive.mount('/content/drive',force_remount=True)

root = "/content/drive/MyDrive/Dr. Moustafa Saad/mine/TFT"

Mounted at /content/drive


In [ ]:
########### Data Downloader ###########
"""
download_epd.py
===============

Download the NHS England Prescribing Dataset (EPD, *with SNOMED code*) monthly
files into a Google Drive folder, ready for `train_epd_tft.py`.

Designed to run in Google Colab:
    1. mounts your Drive (optional),
    2. resolves every monthly file's stable CSV link via the CKAN `package_show`
       API (one metadata call — far cheaper than paginating datastore_search
       over ~17M rows/month),
    3. streams each month to fast local Colab disk, then writes a SLIM, GZIPPED
       copy (only the columns the TFT pipeline needs) into your Drive folder,
    4. is resume-safe: already-downloaded months are skipped, so you can re-run
       after a Colab disconnect.

WHY SLIM BY DEFAULT
    A full EPD month is large; 64 months will not fit on a 15 GB Drive even
    gzipped. The TFT ETL only reads ~11 columns, so `mode="slim_gz"` keeps just
    those and gzips them (~10-20x smaller). Switch to "raw_gz" (all columns,
    gzipped) or "raw" (verbatim CSV) if you want the complete files.

The slim/gzip files are read transparently by `train_epd_tft.build_epd_cache`
(it globs *.csv and *.csv.gz; pandas decompresses .gz automatically).

  ┌─ Colab quick start ────────────────────────────────────────────────────┐
  │ !pip -q install requests tqdm pandas                                    │
  │ from download_epd import main, CONFIG                                   │
  │ CONFIG["drive_dir"] = "/content/drive/MyDrive/epd_run/raw"              │
  │ main()                                                                  │
  └────────────────────────────────────────────────────────────────────────┘
"""

from __future__ import annotations
import os, re, sys, time, gzip, json, shutil, tempfile
from typing import Dict, List, Optional

# ---------------------------------------------------------------------------
# CONFIG  (edit these, or override CONFIG[...] before calling main())
# ---------------------------------------------------------------------------
CONFIG = {
    # inclusive month range "YYYY-MM"
    "start":      "2020-11",
    "end":        "2026-03",

    # where the final files land (a Google Drive path when run in Colab)
    "drive_dir":  os.path.join(root,"EPD"),
    # fast scratch space for the full download before slimming (ephemeral Colab disk)
    "tmp_dir":    "/content/epd_tmp",

    # "slim_gz"  -> only KEEP_COLS, gzipped   (smallest, ETL-ready)  [default]
    # "raw_gz"   -> all columns, gzipped
    # "raw"      -> verbatim CSV, no compression (largest)
    "mode":       "slim_gz",

    "mount_drive":  True,    # call google.colab.drive.mount() if available
    "verify":       True,    # cross-check row counts against the datastore API total
    "overwrite":    False,   # re-download even if the target already exists
    "sleep_between": 1.0,    # politeness delay between files (sec); NHSBSA rate-limits bots
    "max_retries":   4,
    "timeout":       60,
}

# Columns the TFT pipeline consumes (must match EPD_USECOLS in train_epd_tft.py).
KEEP_COLS = [
    "YEAR_MONTH", "REGIONAL_OFFICE_CODE", "ICB_CODE",
    "BNF_CHAPTER_PLUS_CODE", "BNF_CHEMICAL_SUBSTANCE_CODE",
    "ITEMS", "TOTAL_QUANTITY", "NIC", "ACTUAL_COST", "ADQ_USAGE", "UNIDENTIFIED",
]

DATASET_ID = "english-prescribing-dataset-epd-with-snomed-code"
API_BASE = "https://opendata.nhsbsa.net/api/3/action"
PACKAGE_SHOW = f"{API_BASE}/package_show?id={DATASET_ID}"
DATASTORE_SEARCH = API_BASE + "/datastore_search?resource_id={rid}&limit=0"

_YYYYMM_RE = re.compile(r"epd_snomed_(\d{6})", re.IGNORECASE)


# ---------------------------------------------------------------------------
# Pure helpers (no network) — unit-tested below
# ---------------------------------------------------------------------------
def month_range(start: str, end: str) -> List[int]:
    """['2020-11','2026-03'] -> [202011, 202012, ..., 202603] (inclusive)."""
    sy, sm = map(int, start.split("-")); ey, em = map(int, end.split("-"))
    a, b = sy * 12 + (sm - 1), ey * 12 + (em - 1)
    if b < a:
        raise ValueError("end is before start")
    return [(y := divmod(o, 12))[0] * 100 + y[1] + 1 for o in range(a, b + 1)]


def parse_yyyymm(text: str) -> Optional[int]:
    """Extract the 202603-style stamp from a resource url/name, or None."""
    m = _YYYYMM_RE.search(str(text))
    return int(m.group(1)) if m else None


def index_from_package(pkg_json: dict) -> Dict[int, dict]:
    """package_show JSON -> {yyyymm: {"url":..., "rid":..., "name":...}}.
    The CSV `url` is a stable (non-expiring) opendata.nhsbsa.net link."""
    out: Dict[int, dict] = {}
    for r in pkg_json.get("result", {}).get("resources", []):
        url = r.get("url", "") or ""
        ym = parse_yyyymm(url) or parse_yyyymm(r.get("name", ""))
        # prefer the .csv resource; skip non-CSV (e.g. zips) if a CSV exists
        if ym is None:
            continue
        is_csv = url.lower().endswith(".csv")
        if ym not in out or is_csv:
            out[ym] = {"url": url, "rid": r.get("id"), "name": r.get("name", ""),
                       "is_csv": is_csv}
    return out


def target_path(drive_dir: str, ym: int, mode: str) -> str:
    ext = {"slim_gz": "csv.gz", "raw_gz": "csv.gz", "raw": "csv"}[mode]
    return os.path.join(drive_dir, f"epd_snomed_{ym}.{ext}")


def slim_csv_to_gzip(src_csv: str, dst_gz: str, keep_cols: List[str],
                     chunksize: int = 1_000_000) -> int:
    """Stream src CSV in chunks, keep only `keep_cols` that exist, write gzipped.
    Returns the number of data rows written."""
    import pandas as pd
    present: Optional[List[str]] = None
    n_rows = 0
    tmp = dst_gz + ".part"
    with gzip.open(tmp, "wt", newline="", encoding="utf-8") as out:
        reader = pd.read_csv(src_csv, chunksize=chunksize, dtype=str,
                             usecols=lambda c: c in set(keep_cols))
        first = True
        for chunk in reader:
            if present is None:
                present = [c for c in keep_cols if c in chunk.columns]
            chunk = chunk[present]
            chunk.to_csv(out, index=False, header=first)
            n_rows += len(chunk); first = False
    os.replace(tmp, dst_gz)
    return n_rows


# ---------------------------------------------------------------------------
# Network
# ---------------------------------------------------------------------------
def _make_session(max_retries: int):
    import requests
    from requests.adapters import HTTPAdapter
    try:
        from urllib3.util.retry import Retry
        retry = Retry(total=max_retries, backoff_factor=1.5,
                      status_forcelist=(429, 500, 502, 503, 504),
                      allowed_methods=frozenset(["GET"]))
        adapter = HTTPAdapter(max_retries=retry)
    except Exception:
        adapter = HTTPAdapter()
    s = requests.Session()
    s.mount("https://", adapter); s.mount("http://", adapter)
    s.headers.update({"User-Agent": "epd-downloader/1.0 (research)"})
    return s


def _stream_to_file(session, url: str, dst: str, timeout: int) -> int:
    """Download `url` to `dst` with a progress bar. Returns bytes written."""
    from tqdm.auto import tqdm
    tmp = dst + ".part"
    with session.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0)) or None
        written = 0
        bar = tqdm(total=total, unit="B", unit_scale=True,
                   desc=os.path.basename(dst), leave=False)
        with open(tmp, "wb") as f:
            for block in r.iter_content(chunk_size=1 << 20):
                if block:
                    f.write(block); written += len(block); bar.update(len(block))
        bar.close()
    os.replace(tmp, dst)
    return written


def _api_total(session, ym: int, timeout: int) -> Optional[int]:
    """Authoritative row count for a month via datastore_search (limit=0)."""
    try:
        r = session.get(DATASTORE_SEARCH.format(rid=f"EPD_SNOMED_{ym}"), timeout=timeout)
        r.raise_for_status()
        return int(r.json()["result"]["total"])
    except Exception:
        return None


def _count_rows(path: str) -> int:
    op = gzip.open if path.endswith(".gz") else open
    with op(path, "rt", encoding="utf-8", errors="replace") as f:
        return max(0, sum(1 for _ in f) - 1)         # minus header


# ---------------------------------------------------------------------------
# Orchestration
# ---------------------------------------------------------------------------
def _maybe_mount_drive(cfg):
    if not cfg.get("mount_drive"):
        return
    try:
        from google.colab import drive               # type: ignore
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")
    except Exception as e:
        print(f"[drive] not mounting ({e}); using drive_dir as a plain path.")


def main(cfg: Optional[dict] = None):
    cfg = {**CONFIG, **(cfg or {})}
    _maybe_mount_drive(cfg)
    os.makedirs(cfg["drive_dir"], exist_ok=True)
    os.makedirs(cfg["tmp_dir"], exist_ok=True)
    session = _make_session(cfg["max_retries"])

    months = month_range(cfg["start"], cfg["end"])
    print(f"Requested {len(months)} months: {months[0]} … {months[-1]}")

    print("Resolving resource links via package_show …")
    pkg = session.get(PACKAGE_SHOW, timeout=cfg["timeout"]).json()
    index = index_from_package(pkg)
    print(f"  portal exposes {len(index)} EPD-SNOMED monthly files")

    missing = [m for m in months if m not in index]
    if missing:
        print(f"  ⚠ not published / not found: {missing}")

    manifest, ok, skip, fail = [], 0, 0, 0
    for ym in months:
        if ym not in index:
            manifest.append({"month": ym, "status": "missing"}); continue
        dst = target_path(cfg["drive_dir"], ym, cfg["mode"])
        if os.path.exists(dst) and not cfg["overwrite"]:
            print(f"[{ym}] ✓ already present — skipping")
            manifest.append({"month": ym, "status": "skipped",
                             "path": dst, "bytes": os.path.getsize(dst)})
            skip += 1; continue

        url = index[ym]["url"]
        tmp_csv = os.path.join(cfg["tmp_dir"], f"epd_snomed_{ym}.csv")
        try:
            print(f"[{ym}] downloading …")
            _stream_to_file(session, url, tmp_csv, cfg["timeout"])

            if cfg["mode"] == "slim_gz":
                rows = slim_csv_to_gzip(tmp_csv, dst, KEEP_COLS)
            elif cfg["mode"] == "raw_gz":
                with open(tmp_csv, "rb") as fi, gzip.open(dst + ".part", "wb") as fo:
                    shutil.copyfileobj(fi, fo, length=1 << 20)
                os.replace(dst + ".part", dst); rows = _count_rows(dst)
            else:  # raw
                shutil.move(tmp_csv, dst); rows = _count_rows(dst)

            rec = {"month": ym, "status": "ok", "path": dst,
                   "rows": rows, "bytes": os.path.getsize(dst)}

            if cfg["verify"]:
                total = _api_total(session, ym, cfg["timeout"])
                rec["api_total"] = total
                if total is not None and rows != total:
                    rec["status"] = "row_mismatch"
                    print(f"[{ym}] ⚠ rows {rows:,} != API total {total:,}")

            print(f"[{ym}] ✓ {rec['status']}  rows={rows:,}  "
                  f"size={rec['bytes']/1e6:.1f} MB → {dst}")
            manifest.append(rec); ok += 1
        except Exception as e:
            print(f"[{ym}] ✗ failed: {e}")
            manifest.append({"month": ym, "status": "failed", "error": str(e)}); fail += 1
        finally:
            if os.path.exists(tmp_csv):
                try: os.remove(tmp_csv)
                except OSError: pass
            time.sleep(cfg["sleep_between"])

    man_path = os.path.join(cfg["drive_dir"], "_download_manifest.json")
    with open(man_path, "w") as f:
        json.dump(manifest, f, indent=2)
    total_bytes = sum(m.get("bytes", 0) for m in manifest)
    print(f"\nDone. ok={ok} skipped={skip} failed={fail} missing={len(missing)} | "
          f"{total_bytes/1e9:.2f} GB in {cfg['drive_dir']}")
    print(f"Manifest → {man_path}")
    return manifest


if __name__ == "__main__":

    if os.path.exists(os.path.join(root,"EPD")) and not os.listdir(os.path.join(root,"EPD")) :
        main()

In [ ]:
"""
modern_tft.py
=============

A Temporal Fusion Transformer (Lim et al., 2019) rebuilt with a modern
transformer recipe. The classic TFT skeleton is preserved end to end:

    inputs -> per-variable embedding -> Variable Selection Networks (VSN)
           -> static covariate encoders -> LSTM encoder/decoder (locality)
           -> static enrichment -> temporal self-attention
           -> gated skip -> quantile head

Modern optimizations and where each one lives:

  - No bias on linear layers ........ every nn.Linear uses bias=False;
                                       continuous embeddings are a pure outer
                                       product (no offset).
  - Clean residual path ............. blocks add raw x; norms only ever sit on
                                       the sublayer output, never on the skip.
  - RMSNorm before AND after ........ "sandwich" norm:
                                       post_norm(sublayer(pre_norm(x))).
  - Gated activations ............... GeGLU FFN in the transformer block; GRNs
                                       are themselves gated (GLU/GeGLU/ReGLU).
  - Parallel blocks, summed ......... out = x + attn(...) + ffn(...).
  - RoPE ............................ rotary position embedding on Q, K.
  - d_ff = 2.5 * d_model ............ set in TFTConfig.__post_init__.
  - Aspect-ratio sizing ............. recommend_param_budget() helper.
  - dropout=0.1, wd=0.1 + cosine .... defaults + build_optimizer (decoupled wd,
                                       param groups) + cosine-with-warmup.
  - QK-norm ......................... RMSNorm on Q and K (per head) pre-dot.
  - GQA / MQA ....................... n_kv_heads < n_heads => grouped queries;
                                       n_kv_heads == 1 => multi-query.
  - Grad checkpoint + accumulation .. checkpoint on blocks; accumulation in loop.
  - Flash attention ................. F.scaled_dot_product_attention. For TFT's
                                       interpretable attention maps, flip
                                       need_weights=True to use the eager path.
  - Mixed precision ................. autocast (+ GradScaler for fp16) in loop.
  - torch.compile ................... operator fusion, memory-coalesced layouts,
                                       and automatic Triton lowering.
  - Optional Triton ................. hand-written RMSNorm forward kernel used at
                                       inference, as an illustration.
"""

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
!pip install triton
# ---------------------------------------------------------------------------
# Optional Triton. torch.compile already emits Triton for the whole model; this
# hand-written forward kernel is an illustration, used only at inference time
# (no custom backward -> no autograd through it).
# ---------------------------------------------------------------------------
try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
except Exception:  # pragma: no cover - triton is optional
    HAS_TRITON = False


# ===========================================================================
# Config
# ===========================================================================
@dataclass
class TFTConfig:
    # ---- data schema -------------------------------------------------------
    static_cat_cardinalities: tuple[int, ...] = ()    # categorical static covariates
    n_static_real: int = 0                            # continuous static covariates
    known_cat_cardinalities: tuple[int, ...] = ()     # known-future categoricals
    n_known_real: int = 0                             # known-future continuous
    observed_cat_cardinalities: tuple[int, ...] = ()  # past-only categoricals
    n_observed_real: int = 1                          # past-only continuous (incl. target history)

    # ---- model dims --------------------------------------------------------
    d_model: int = 64
    n_heads: int = 4
    n_kv_heads: int = 1            # GQA: < n_heads => grouped; == 1 => MQA
    n_blocks: int = 1             # TFT classically uses 1 attention layer
    d_ff: Optional[int] = None    # defaults to round(2.5 * d_model)
    dropout: float = 0.1

    # ---- attention ---------------------------------------------------------
    rope_base: float = 10_000.0
    qk_norm: bool = True
    use_flash: bool = True        # SDPA fast path; False => eager + weights
    parallel_blocks: bool = True
    grad_checkpoint: bool = False
    max_len: int = 4096           # RoPE table size

    # ---- gating ------------------------------------------------------------
    ffn_gate: str = "reglu"       # block FFN: "geglu" | "reglu" | "swiglu"
    grn_gate: str = "reglu"         # GRN gate: "glu" | "geglu" | "reglu"

    # ---- head --------------------------------------------------------------
    quantiles: tuple[float, ...] = (0.1, 0.5, 0.9)

    def __post_init__(self):
        if self.d_ff is None:
            self.d_ff = round(4.0 * self.d_model)
        assert self.n_heads % self.n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        assert self.d_model % self.n_heads == 0, "d_model must be divisible by n_heads"

    @property
    def head_dim(self) -> int:
        return self.d_model // self.n_heads


# ===========================================================================
# Norm (RMSNorm, with optional Triton inference kernel + QK-norm reuse)
# ===========================================================================
if HAS_TRITON:
    @triton.jit
    def _rmsnorm_fwd_kernel(X, W, Y, stride, N, eps, BLOCK: tl.constexpr):
        row = tl.program_id(0)
        X += row * stride
        Y += row * stride
        cols = tl.arange(0, BLOCK)
        mask = cols < N
        x = tl.load(X + cols, mask=mask, other=0.0).to(tl.float32)
        var = tl.sum(x * x, axis=0) / N
        rstd = 1.0 / tl.sqrt(var + eps)
        w = tl.load(W + cols, mask=mask, other=0.0).to(tl.float32)
        tl.store(Y + cols, x * rstd * w, mask=mask)

    def _rmsnorm_triton(x: torch.Tensor, weight: torch.Tensor, eps: float) -> torch.Tensor:
        N = x.shape[-1]
        x2 = x.reshape(-1, N).contiguous()
        y = torch.empty_like(x2)
        BLOCK = triton.next_power_of_2(N)
        _rmsnorm_fwd_kernel[(x2.shape[0],)](x2, weight, y, x2.stride(0), N, eps, BLOCK=BLOCK)
        return y.reshape(x.shape)


class RMSNorm(nn.Module):
    """Root-mean-square norm. On CUDA at inference it uses the Triton kernel;
    during training it stays in PyTorch so torch.compile can fuse it."""

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if HAS_TRITON and x.is_cuda and not torch.is_grad_enabled():
            return _rmsnorm_triton(x, self.weight, self.eps)
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


# ===========================================================================
# RoPE
# ===========================================================================
def _rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


class RoPE(nn.Module):
    def __init__(self, head_dim: int, base: float, max_len: int):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        t = torch.arange(max_len).float()
        freqs = torch.outer(t, inv_freq)             # (max_len, hd/2)
        emb = torch.cat((freqs, freqs), dim=-1)      # (max_len, hd)
        self.register_buffer("cos", emb.cos(), persistent=False)
        self.register_buffer("sin", emb.sin(), persistent=False)

    def forward(self, seq_len: int):
        return self.cos[:seq_len], self.sin[:seq_len]


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    # x: (B, H, T, hd) ; cos/sin: (T, hd)
    return x * cos[None, None] + _rotate_half(x) * sin[None, None]


# ===========================================================================
# Gated FFN -- GeGLU / ReGLU / SwiGLU
# ===========================================================================
class GatedFFN(nn.Module):
    def __init__(self, d_model: int, d_ff: int, gate: str = "geglu", dropout: float = 0.0):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
        self.act = {"geglu": F.gelu, "reglu": F.relu, "swiglu": F.silu}[gate]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w_down(self.drop(self.act(self.w_gate(x)) * self.w_up(x)))


# ===========================================================================
# Attention -- GQA + RoPE + QK-norm; Flash fast path, eager interpretable path
# ===========================================================================
class Attention(nn.Module):
    def __init__(self, cfg: TFTConfig):
        super().__init__()
        self.cfg = cfg
        self.nh = cfg.n_heads
        self.nkv = cfg.n_kv_heads
        self.hd = cfg.head_dim
        self.rep = self.nh // self.nkv

        self.q_proj = nn.Linear(cfg.d_model, self.nh * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.nh * self.hd, cfg.d_model, bias=False)

        self.q_norm = RMSNorm(self.hd) if cfg.qk_norm else nn.Identity()
        self.k_norm = RMSNorm(self.hd) if cfg.qk_norm else nn.Identity()
        self.attn_drop = cfg.dropout

    def forward(self, x, cos, sin, need_weights: bool = False):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.nh, self.hd).transpose(1, 2)   # (B,nh,T,hd)
        k = self.k_proj(x).view(B, T, self.nkv, self.hd).transpose(1, 2)  # (B,nkv,T,hd)
        v = self.v_proj(x).view(B, T, self.nkv, self.hd).transpose(1, 2)

        q = self.q_norm(q)               # norm BEFORE the dot product
        k = self.k_norm(k)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # expand KV groups -> n_heads (storage/cache stays at nkv heads)
        k = k.repeat_interleave(self.rep, dim=1)
        v = v.repeat_interleave(self.rep, dim=1)

        if self.cfg.use_flash and not need_weights:
            out = F.scaled_dot_product_attention(
                q, k, v, is_causal=True,
                dropout_p=self.attn_drop if self.training else 0.0,
            )
            weights = None
        else:
            scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.hd)
            causal = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()
            scores = scores.masked_fill(~causal, float("-inf"))
            attn = scores.softmax(dim=-1)
            attn = F.dropout(attn, p=self.attn_drop, training=self.training)
            out = attn @ v
            weights = attn.mean(dim=1)   # (B,T,T) interpretable map (avg heads)

        out = out.transpose(1, 2).reshape(B, T, self.nh * self.hd)
        return self.o_proj(out), weights


# ===========================================================================
# Modern transformer block -- sandwich norm, clean residual, parallel
# ===========================================================================
class TransformerBlock(nn.Module):
    def __init__(self, cfg: TFTConfig):
        super().__init__()
        self.cfg = cfg
        self.attn = Attention(cfg)
        self.ffn = GatedFFN(cfg.d_model, cfg.d_ff, cfg.ffn_gate, cfg.dropout)
        self.pre_attn = RMSNorm(cfg.d_model)
        self.post_attn = RMSNorm(cfg.d_model)
        self.pre_ffn = RMSNorm(cfg.d_model)
        self.post_ffn = RMSNorm(cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)

    def _attn(self, x, cos, sin, need_weights):
        a, w = self.attn(self.pre_attn(x), cos, sin, need_weights)
        return self.drop(self.post_attn(a)), w

    def _ffn(self, x):
        return self.drop(self.post_ffn(self.ffn(self.pre_ffn(x))))

    def forward(self, x, cos, sin, need_weights: bool = False):
        use_ckpt = self.cfg.grad_checkpoint and self.training and not need_weights
        if self.cfg.parallel_blocks:
            if use_ckpt:
                a, _ = checkpoint(self._attn, x, cos, sin, False, use_reentrant=False)
                f = checkpoint(self._ffn, x, use_reentrant=False)
                w = None
            else:
                a, w = self._attn(x, cos, sin, need_weights)
                f = self._ffn(x)
            return x + a + f, w          # clean residual: raw x
        else:
            a, w = self._attn(x, cos, sin, need_weights)
            x = x + a
            x = x + self._ffn(x)
            return x, w


# ===========================================================================
# TFT building blocks: GRN, embeddings, variable selection, gated skip
# ===========================================================================
class GRN(nn.Module):
    """Gated Residual Network -- TFT's workhorse. RMSNorm, no bias, gated."""

    def __init__(self, in_dim, hidden, out_dim, ctx_dim=None, dropout=0.1, gate="glu"):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden, bias=False)
        self.ctx = nn.Linear(ctx_dim, hidden, bias=False) if ctx_dim else None
        self.fc2 = nn.Linear(hidden, hidden, bias=False)
        self.gate = nn.Linear(hidden, out_dim, bias=False)
        self.val = nn.Linear(hidden, out_dim, bias=False)
        self.skip = nn.Identity() if in_dim == out_dim else nn.Linear(in_dim, out_dim, bias=False)
        self.norm = RMSNorm(out_dim)
        self.drop = nn.Dropout(dropout)
        self._gate_act = {"glu": torch.sigmoid, "geglu": F.gelu, "reglu": F.relu}[gate]

    def forward(self, a, c=None):
        h = self.fc1(a)
        if self.ctx is not None and c is not None:
            h = h + self.ctx(c)
        h = F.elu(h)
        h = self.drop(self.fc2(h))
        gated = self.val(h) * self._gate_act(self.gate(h))
        return self.norm(self.skip(a) + gated)


class ContinuousEmbedding(nn.Module):
    """Per-variable scalar -> d_model. Pure outer product, no bias."""

    def __init__(self, n_vars: int, d_model: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_vars, d_model) * 0.02)

    def forward(self, x):                            # x: (..., n_vars)
        return x.unsqueeze(-1) * self.weight         # (..., n_vars, d_model)


class CategoricalEmbedding(nn.Module):
    def __init__(self, cardinalities, d_model: int):
        super().__init__()
        self.embs = nn.ModuleList(nn.Embedding(c, d_model) for c in cardinalities)

    def forward(self, x):                            # x: (..., n_cat) long
        if len(self.embs) == 0:
            return x.new_zeros(*x.shape[:-1], 0, 0)
        return torch.stack([e(x[..., i]) for i, e in enumerate(self.embs)], dim=-2)


class VariableSelectionNetwork(nn.Module):
    """Soft variable selection: per-variable GRN + softmax weights."""

    def __init__(self, n_vars: int, d_model: int, ctx_dim: Optional[int], dropout: float, gate: str):
        super().__init__()
        self.n_vars = n_vars
        self.flatten_grn = GRN(n_vars * d_model, d_model, n_vars, ctx_dim, dropout, gate)
        self.var_grns = nn.ModuleList(
            GRN(d_model, d_model, d_model, None, dropout, gate) for _ in range(n_vars)
        )

    def forward(self, var_embeds: torch.Tensor, ctx=None):
        flat = var_embeds.flatten(start_dim=-2)                # (..., n_vars*d_model)
        weights = self.flatten_grn(flat, ctx).softmax(dim=-1)  # (..., n_vars)
        processed = torch.stack(
            [grn(var_embeds[..., i, :]) for i, grn in enumerate(self.var_grns)], dim=-2
        )                                                      # (..., n_vars, d_model)
        out = (weights.unsqueeze(-1) * processed).sum(dim=-2)  # (..., d_model)
        return out, weights


class GateAddNorm(nn.Module):
    """GLU gated skip connection + RMSNorm (TFT's gate-add-norm), modernized."""

    def __init__(self, d_model: int, dropout: float):
        super().__init__()
        self.gate = nn.Linear(d_model, d_model, bias=False)
        self.val = nn.Linear(d_model, d_model, bias=False)
        self.norm = RMSNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, residual):
        x = self.drop(x)
        x = self.val(x) * torch.sigmoid(self.gate(x))
        return self.norm(x + residual)


# ===========================================================================
# The model
# ===========================================================================
class ModernTFT(nn.Module):
    def __init__(self, cfg: TFTConfig):
        super().__init__()
        self.cfg = cfg
        d = cfg.d_model

        # ---- embeddings -----------------------------------------------------
        self.static_cat_emb = CategoricalEmbedding(cfg.static_cat_cardinalities, d)
        self.static_real_emb = ContinuousEmbedding(cfg.n_static_real, d) if cfg.n_static_real else None
        self.known_cat_emb = CategoricalEmbedding(cfg.known_cat_cardinalities, d)
        self.known_real_emb = ContinuousEmbedding(cfg.n_known_real, d) if cfg.n_known_real else None
        self.obs_cat_emb = CategoricalEmbedding(cfg.observed_cat_cardinalities, d)
        self.obs_real_emb = ContinuousEmbedding(cfg.n_observed_real, d) if cfg.n_observed_real else None

        n_static = len(cfg.static_cat_cardinalities) + cfg.n_static_real
        n_known = len(cfg.known_cat_cardinalities) + cfg.n_known_real
        n_obs = len(cfg.observed_cat_cardinalities) + cfg.n_observed_real
        assert n_static >= 1 and n_known >= 1 and (n_known + n_obs) >= 1

        # ---- variable selection --------------------------------------------
        self.vsn_static = VariableSelectionNetwork(n_static, d, None, cfg.dropout, cfg.grn_gate)
        self.vsn_encoder = VariableSelectionNetwork(n_known + n_obs, d, d, cfg.dropout, cfg.grn_gate)
        self.vsn_decoder = VariableSelectionNetwork(n_known, d, d, cfg.dropout, cfg.grn_gate)

        # ---- static covariate encoders (4 contexts) ------------------------
        self.grn_select = GRN(d, d, d, None, cfg.dropout, cfg.grn_gate)   # c_s
        self.grn_enrich = GRN(d, d, d, None, cfg.dropout, cfg.grn_gate)   # c_c
        self.grn_lstm_h = GRN(d, d, d, None, cfg.dropout, cfg.grn_gate)   # h0
        self.grn_lstm_c = GRN(d, d, d, None, cfg.dropout, cfg.grn_gate)   # c0

        # ---- LSTM enc/dec (locality enhancement) ---------------------------
        self.enc_lstm = nn.LSTM(d, d, batch_first=True)
        self.dec_lstm = nn.LSTM(d, d, batch_first=True)
        self.lstm_gate = GateAddNorm(d, cfg.dropout)

        # ---- static enrichment ---------------------------------------------
        self.enrich_grn = GRN(d, d, d, d, cfg.dropout, cfg.grn_gate)

        # ---- temporal self-attention (modern blocks) ----------------------
        self.rope = RoPE(cfg.head_dim, cfg.rope_base, cfg.max_len)
        self.blocks = nn.ModuleList(TransformerBlock(cfg) for _ in range(cfg.n_blocks))
        self.attn_gate = GateAddNorm(d, cfg.dropout)

        # ---- quantile head --------------------------------------------------
        self.head = nn.Linear(d, len(cfg.quantiles), bias=False)

    def _gather(self, cat_emb, cat_x, real_emb, real_x):
        parts = []
        if cat_x is not None and cat_x.shape[-1] > 0:
            parts.append(cat_emb(cat_x))
        if real_x is not None and real_emb is not None and real_x.shape[-1] > 0:
            parts.append(real_emb(real_x))
        return torch.cat(parts, dim=-2)

    def forward(self, batch: dict, need_weights: bool = False):
        """
        batch keys:
            static_cat   : (B, n_static_cat)        long
            static_real  : (B, n_static_real)       float
            known_cat    : (B, T, n_known_cat)      long   (T = L + H)
            known_real   : (B, T, n_known_real)     float
            observed_cat : (B, L, n_observed_cat)   long
            observed_real: (B, L, n_observed_real)  float
        """
        known_real = batch.get("known_real")
        known_cat = batch.get("known_cat")
        T = (known_real if known_real is not None else known_cat).shape[1]
        obs = batch.get("observed_real")
        obs = obs if obs is not None else batch.get("observed_cat")
        L = obs.shape[1]
        H = T - L

        # ---- static -> selection + 4 contexts ------------------------------
        static_vars = self._gather(self.static_cat_emb, batch.get("static_cat"),
                                   self.static_real_emb, batch.get("static_real"))
        static_vec, static_w = self.vsn_static(static_vars)
        c_s = self.grn_select(static_vec)
        c_c = self.grn_enrich(static_vec)
        h0 = self.grn_lstm_h(static_vec).unsqueeze(0)
        c0 = self.grn_lstm_c(static_vec).unsqueeze(0)

        # ---- embeddings ----------------------------------------------------
        known_vars = self._gather(self.known_cat_emb, known_cat, self.known_real_emb, known_real)
        obs_vars = self._gather(self.obs_cat_emb, batch.get("observed_cat"),
                                self.obs_real_emb, batch.get("observed_real"))

        # ---- encoder selection (past) --------------------------------------
        enc_vars = torch.cat([known_vars[:, :L], obs_vars], dim=-2)
        enc_feat, enc_w = self.vsn_encoder(enc_vars, c_s.unsqueeze(1).expand(-1, L, -1))

        # ---- decoder selection (future) ------------------------------------
        dec_vars = known_vars[:, L:]
        dec_feat, dec_w = self.vsn_decoder(dec_vars, c_s.unsqueeze(1).expand(-1, H, -1))

        # ---- LSTM enc/dec + gated skip -------------------------------------
        enc_out, (hn, cn) = self.enc_lstm(enc_feat, (h0, c0))
        dec_out, _ = self.dec_lstm(dec_feat, (hn, cn))
        lstm_out = torch.cat([enc_out, dec_out], dim=1)
        vsn_feat = torch.cat([enc_feat, dec_feat], dim=1)
        temporal = self.lstm_gate(lstm_out, vsn_feat)

        # ---- static enrichment ---------------------------------------------
        enriched = self.enrich_grn(temporal, c_c.unsqueeze(1).expand(-1, T, -1))

        # ---- temporal self-attention ---------------------------------------
        cos, sin = self.rope(T)
        x = enriched
        attn_map = None
        for blk in self.blocks:
            x, w = blk(x, cos, sin, need_weights)
            if w is not None:
                attn_map = w
        attended = self.attn_gate(x, temporal)

        # ---- output (decoder steps only) -----------------------------------
        logits = self.head(attended[:, L:])          # (B, H, Q)

        out = {"prediction": logits}
        if need_weights:
            out.update(static_weights=static_w, encoder_weights=enc_w,
                       decoder_weights=dec_w, attention=attn_map)
        return out


# ===========================================================================
# Loss
# ===========================================================================
def quantile_loss(preds: torch.Tensor, target: torch.Tensor, quantiles) -> torch.Tensor:
    """Pinball / quantile loss. preds: (B,H,Q), target: (B,H)."""
    q = torch.as_tensor(quantiles, device=preds.device, dtype=preds.dtype)
    errors = target.unsqueeze(-1) - preds
    return torch.maximum(q * errors, (q - 1.0) * errors).mean()


# ===========================================================================
# Optimization helpers
# ===========================================================================
def recommend_param_budget(n_tokens: int, ratio: int = 150) -> float:
    """'Aspect ratio' sizing: ~100-200 tokens per parameter. For a numeric TFT,
    treat (timesteps x active features) as a rough token proxy; use as a sanity
    bound, not a hard target."""
    return n_tokens / ratio


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def build_optimizer(model: nn.Module, lr: float = 3e-4, weight_decay: float = 0.1):
    """Decoupled weight decay (AdamW). Norm gains / embeddings excluded; decay
    acts as an optimizer complement on the matmul weights."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (no_decay if (p.ndim < 2 or "emb" in name.lower()) else decay).append(p)
    groups = [
        {"params": decay, "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]
    return torch.optim.AdamW(groups, lr=lr, betas=(0.9, 0.95), fused=torch.cuda.is_available())


def cosine_warmup_schedule(optimizer, warmup_steps: int, total_steps: int, min_ratio: float = 0.1):
    """Cosine annealing with linear warmup."""
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def compile_model(model: nn.Module, **kwargs) -> nn.Module:
    """torch.compile: operator fusion, memory-coalesced layouts, Triton lowering."""
    return torch.compile(model, **kwargs)


# ===========================================================================
# Training step (grad checkpoint + accumulation, mixed precision)
# ===========================================================================
def train_epoch(model, loader, optimizer, scheduler, cfg: TFTConfig,
                device="cuda", accum_steps: int = 1, amp_dtype=torch.bfloat16):
    model.train()
    use_scaler = (amp_dtype == torch.float16) and device.startswith("cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_scaler)
    optimizer.zero_grad(set_to_none=True)

    for i, batch in enumerate(loader):
        batch = {k: v.to(device) for k, v in batch.items() if torch.is_tensor(v)}
        target = batch.pop("target")
        with torch.autocast(device_type=device.split(":")[0], dtype=amp_dtype):
            out = model(batch)
            loss = quantile_loss(out["prediction"], target, cfg.quantiles) / accum_steps

        scaler.scale(loss).backward() if use_scaler else loss.backward()

        if (i + 1) % accum_steps == 0:
            if use_scaler:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if use_scaler:
                scaler.step(optimizer); scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()


# ===========================================================================
# Smoke test (CPU, eager): forward + backward + interpretability path
# ===========================================================================
if __name__ == "__main__":
    torch.manual_seed(0)
    cfg = TFTConfig(
        static_cat_cardinalities=(5, 3), n_static_real=2,
        known_cat_cardinalities=(7,), n_known_real=2,
        observed_cat_cardinalities=(), n_observed_real=3,
        d_model=32, n_heads=4, n_kv_heads=2, n_blocks=2,
        dropout=0.1, qk_norm=True, use_flash=True,
        parallel_blocks=True, grad_checkpoint=True,
        quantiles=(0.1, 0.5, 0.9),
    )
    B, L, H = 8, 24, 6
    T = L + H
    model = ModernTFT(cfg)
    batch = dict(
        static_cat=torch.randint(0, 3, (B, 2)),
        static_real=torch.randn(B, 2),
        known_cat=torch.randint(0, 7, (B, T, 1)),
        known_real=torch.randn(B, T, 2),
        observed_real=torch.randn(B, L, 3),
    )
    target = torch.randn(B, H)

    print(f"params: {count_params(model):,}")
    print(f"d_ff (2.5*d_model): {cfg.d_ff}")

    out = model(batch)
    loss = quantile_loss(out["prediction"], target, cfg.quantiles)
    loss.backward()
    print(f"prediction (B,H,Q): {tuple(out['prediction'].shape)} | loss {loss.item():.4f} | backward OK")

    model.eval()
    with torch.no_grad():
        interp = model(batch, need_weights=True)
    print(f"attention {tuple(interp['attention'].shape)} | "
          f"static sel {tuple(interp['static_weights'].shape)} | "
          f"encoder sel {tuple(interp['encoder_weights'].shape)}")

    opt = build_optimizer(model)
    sch = cosine_warmup_schedule(opt, 10, 100)
    print(f"optimizer groups: {len(opt.param_groups)} (decay / no-decay) | OK")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 4.8 MB/s eta 0:00:00
params: 149,421
d_ff (2.5*d_model): 128
prediction (B,H,Q): (8, 6, 3) | loss 0.4590 | backward OK
attention (8, 30, 30) | static sel (8, 4) | encoder sel (8, 24, 6)
optimizer groups: 2 (decay / no-decay) | OK


In [ ]:
"""
train_epd_tft.py
================

NHS England Prescribing Dataset (EPD) monthly drug-demand forecasting with the
ModernTFT model (see modern_tft.py), wrapped in the same trainer idiom as the
a CONFIG dict + a single Trainer class that owns
datasets, loaders, model, loss, optimizer, SequentialLR (warmup -> cosine) with
a warm restart, AMP, gradient accumulation, history JSON, checkpointing, plots,
and a synthetic fallback so the whole thing is runnable before the (open) raw
files are downloaded.


KEY DESIGN CHANGES vs the MIMIC pipeline
----------------------------------------
  series          one ICU stay (relative time)      -> one (ICB x BNF-chemical) monthly series (real calendar)
  time axis       hours since intime                 -> global month index (shared across all series)
  target          a vital (z-scored)                 -> ITEMS dispensed, log1p then z-scored (count demand)
  static_cat      gender/careunit/admtype/race       -> region / ICB / BNF-chapter / BNF-chemical
  static_real     age                                -> series log-mean & log-std of demand (scale + volatility)
  known (future)  hour-of-day, elapsed frac          -> month-of-year (12), sin/cos month, linear time-trend
  observed (past) vitals incl. target                -> log-demand (target) + cost/item + qty/item + ADQ/item
  split           random by stay                     -> TEMPORAL (train past / val & test future)  [no leakage]
  metric denorm   affine z -> physical               -> expm1 of (z * std + mean) ; adds WAPE (demand metric)

DATA CONTRACT (what the dataset yields, matching ModernTFT.forward):
    static_cat   : (B, n_static_cat)       long    region, ICB, chapter, chemical
    static_real  : (B, n_static_real)      float   series log-mean & log-std demand
    known_cat    : (B, T, 1)               long    month-of-year (0..11)
    known_real   : (B, T, 3)               float   sin(month), cos(month), time-trend
    observed_real: (B, L, n_observed_real) float   [log-demand(target), cost/item, qty/item, adq/item]
    target       : (B, H)                  float   future log-demand (z-scored, channel 0)
  where T = L (encoder/past months) + H (decoder/horizon months).
"""

from __future__ import annotations
import os, sys, json, time, math, glob
from pathlib import Path
from typing import Optional, Dict, List, Tuple
from dataclasses import asdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    HAS_MPL = False

In [ ]:
# =============================================================================
# CONFIG
# =============================================================================
CONFIG = {
    # ── Paths ───────────────────────────────────────────────────────────
    # Point `epd_raw` at a directory of monthly EPD CSVs (or .csv.gz), e.g.
    # EPD_202401.csv ... downloaded from the NHSBSA Open Data Portal:
    #   https://opendata.nhsbsa.net/dataset/english-prescribing-dataset-epd-with-snomed-code
    # (open data, no credentialing — unlike MIMIC). The ETL streams them in
    # chunks; aggregated monthly series are tiny.
    "epd_raw":        os.path.join(root, "EPD"),
    "cache_dir":      os.path.join(root, "cache","44"),
    "splits_dir":     os.path.join(root, "cache", "splits"),
    "logdir":         os.path.join(root, "tft_epd_forecast_44", "logs"),
    "resume":         os.path.join(root, "tft_epd_forecast_44", "checkpoints", "best.pth"),
    "submission_csv": os.path.join(root, "tft_epd_forecast_44", "forecasts.csv"),

    # ── Series definition / cohort ──────────────────────────────────────
    "series_keys":    ["REGIONAL_OFFICE_CODE", "BNF_CHAPTER_PLUS_CODE"],  # one series per (ICB, chemical)
    "target_col":     "ITEMS",            # demand target (number of prescription items)
    "bnf_chapter_filter": None,           # e.g. "04: Central Nervous System" to focus one chapter; None = all
    "min_active_months": 36,              # drop series with too little history to window
    "top_k_series":   None,               # keep only the K highest-volume series (None = all)
    # NHS July-2022 reorg: STPs/CCGs -> ICBs/Sub-ICBs. Codes/structure pre-Jul-2022
    # were NOT retrospectively updated, so ICB-keyed series break at that boundary.
    # Either harmonize legacy codes onto ICB codes via a crosswalk, or floor the
    # window to 2022-07 so every series lives entirely in the ICB era.
    "geo_crosswalk":  None,               # {"ICB_CODE": {legacy_code: icb_code, ...}}  (identity if None)
    "min_year_month": None,               # e.g. "2022-07" to start after the reorg
    "max_year_month": None,               # e.g. "2026-03"

    # ── Task / windowing (MONTHLY) ──────────────────────────────────────
    "encoder_len":    12,                 # L months of history (2 seasonal cycles)
    "horizon":        6,                  # H months ahead
    "stride":         1,                  # window stride (months)
    # Temporal split: the LAST `test_months` of calendar are test horizons,
    # the `val_months` before that are val horizons, everything earlier is train.
    "val_months":     6,
    "test_months":    12,

    # ── Model (ModernTFT) ───────────────────────────────────────────────
    "d_model":        64,
    "n_heads":        4,
    "n_kv_heads":     1,
    "n_blocks":       2,
    "dropout":        0.1,
    "quantiles":      (0.1, 0.25, 0.5, 0.75, 0.9),

    # ── Optim ───────────────────────────────────────────────────────────
    "batch_size":     64,
    "num_workers":    2,
    "lr":             1e-3,
    "weight_decay":   0.1,
    "feature_lr_mult": 0.5,               # slow group (embeds/LSTM/VSN/GRN) LR multiplier
    "num_epochs":     100,
    "warmup":         5,
    "restart_epoch":  40,
    "restart_lr":     5e-5,
    "accumulation":   1,
    "use_amp":        True,
    "grad_clip":      1.0,

    # ── Ramped aux loss (quantile non-crossing penalty) ─────────────────
    "aux_weight_max":      0.1,
    "aux_ramp_start_epoch": 5,
    "aux_ramp_end_epoch":   20,

    # ── Misc ────────────────────────────────────────────────────────────
    "train_metric_stride": 4,
    "early_stopping_patience": 15,
    "early_stopping_min_delta": 1e-4,
    "device":         "cuda" if torch.cuda.is_available() else "cpu",
    "is_train":       True,
    "seed":           44,
}

# Raw EPD columns we read (matches the NHSBSA schema in the sample record).
EPD_USECOLS = [
    "YEAR_MONTH", "REGIONAL_OFFICE_CODE", "ICB_CODE",
    "BNF_CHAPTER_PLUS_CODE", "BNF_CHEMICAL_SUBSTANCE_CODE",
    "ITEMS", "TOTAL_QUANTITY", "NIC", "ACTUAL_COST", "ADQ_USAGE", "UNIDENTIFIED",
]
# Static categoricals describing a series (parents of the series keys included).
STATIC_CAT_COLS = ["REGIONAL_OFFICE_CODE", "BNF_CHAPTER_PLUS_CODE"]
AGG_COLS = ["ITEMS", "TOTAL_QUANTITY", "NIC", "ACTUAL_COST"]

# =============================================================================
# Small utilities (mirror the BirdCLEF helpers)
# =============================================================================
class TeeFile:
    """Write to a base stream AND a log file simultaneously. Used two ways:
    as the tqdm `file=` target, and (via redirect_stdout) to capture bare
    print()s, so the on-disk logs match the console exactly — no manual copying.

    Line-buffered + utf-8 so the file is populated live (survives a Colab
    disconnect) and tolerates the box-drawing chars in the banners."""
    def __init__(self, stream, path):
        self.stream = stream
        self.log = open(path, "a", buffering=1, encoding="utf-8", errors="replace")
    def write(self, data):
        self.stream.write(data); self.log.write(data)
        return len(data)
    def flush(self):
        try: self.stream.flush()
        except Exception: pass
        try: self.log.flush()
        except Exception: pass
    def isatty(self):
        return getattr(self.stream, "isatty", lambda: False)()
    def close(self):
        try: self.log.close()
        except Exception: pass


# tqdm is optional; degrade to a no-op wrapper if missing.
# IMPORTANT: use the PLAIN tqdm, not tqdm.auto. In a notebook, tqdm.auto renders
# to the ipywidgets backend, which IGNORES the `file=` argument — that is why the
# train/val logs came out empty and had to be copied by hand. Plain tqdm honors
# `file=`, so the bars land in the log files as intended.
try:
    from tqdm import tqdm
except Exception:                                       # pragma: no cover
    def tqdm(it=None, **k):
        return it if it is not None else []


class EarlyStopping:
    def __init__(self, patience=15, min_delta=1e-4):
        self.patience = patience; self.min_delta = min_delta
        self.best = math.inf; self.count = 0; self.stop = False
    def step(self, value):                 # lower is better
        if value < self.best - self.min_delta:
            self.best = value; self.count = 0
        else:
            self.count += 1
            if self.count >= self.patience:
                self.stop = True
        return self.stop


def parse_year_month(v) -> int:
    """EPD YEAR_MONTH -> absolute month ordinal (year*12 + month-1).
    Handles 'YYYY-MM', 'YYYYMM', and integer 202603."""
    s = str(v).strip()
    if "-" in s:
        y, m = s.split("-")[:2]
        y, m = int(y), int(m)
    else:
        iv = int(float(s)); y, m = iv // 100, iv % 100
    return y * 12 + (m - 1)


def ord_to_label(ordm: int) -> str:
    y, m = divmod(ordm, 12)
    return f"{y}-{m + 1:02d}"


# =============================================================================
# EPD  ->  cache  (real ETL + synthetic fallback)
# =============================================================================
def _list_epd_files(raw_dir: str) -> List[str]:
    files = sorted(glob.glob(os.path.join(raw_dir, "*.csv")) +
                   glob.glob(os.path.join(raw_dir, "*.csv.gz")))
    return files


# tokens that mark a row's practice/geography as unidentified (EPD guidance).
_UNIDENT_TRUE = {"Y", "YES", "TRUE", "T", "1"}
_UNIDENT_SENTINELS = {"", "UNIDENTIFIED", "NONE", "NAN"}


def build_epd_cache(cfg):
    """Real ETL from raw monthly EPD CSVs. Streams each file in chunks, drops
    unidentified-practice rows, harmonizes geography (NHS July-2022 STP->ICB
    reorg, via cfg['geo_crosswalk']), and sums the demand channels per
    (series, month). Summing across the finer SNOMED rows (multiple SNOMED codes
    per BNF presentation) up to the chosen series key is exactly the collapse the
    EPD release guidance recommends ('group by a higher element, do not include
    SNOMED_CODE')."""
    raw = cfg["epd_raw"]; files = _list_epd_files(raw)
    if not files:
        raise FileNotFoundError(
            f"No EPD CSVs under {raw}. Download monthly files from the NHSBSA "
            f"Open Data Portal, or call prepare_synthetic_epd_cache(CONFIG).")

    keys = cfg["series_keys"]; chap = cfg.get("bnf_chapter_filter")
    crosswalk = cfg.get("geo_crosswalk")           # {col: {old_code: new_code}}
    ym_lo = parse_year_month(cfg["min_year_month"]) if cfg.get("min_year_month") else None
    ym_hi = parse_year_month(cfg["max_year_month"]) if cfg.get("max_year_month") else None
    gb_cols = list(dict.fromkeys(keys + STATIC_CAT_COLS + ["ym"]))

    parts = []
    for fp in files:
        head = pd.read_csv(fp, nrows=0)
        # Pre-July-2022 files carry the legacy STP/CCG structure and may lack the
        # ICB columns entirely; warn and skip rather than crash (or supply a
        # crosswalk + rename upstream so these months are usable).
        missing = [c for c in (keys + ["UNIDENTIFIED"]) if c not in head.columns]
        if missing:
            print(f"  [skip] {os.path.basename(fp)}: missing columns {missing} "
                  f"(likely a pre-reorg schema — see July-2022 note).")
            continue

        reader = pd.read_csv(fp, usecols=lambda c: c in EPD_USECOLS,
                             chunksize=1_000_000, dtype={"YEAR_MONTH": str})
        for chunk in tqdm(reader, desc=f"{os.path.basename(fp)}"):
            for _c in AGG_COLS:                 # files lack ADQ_USAGE; keep schema intact
                if _c not in chunk.columns:
                    chunk[_c] = 0.0
            # ── drop unidentified-practice rows (robust to Y/N *and* TRUE/FALSE) ──
            unident = chunk["UNIDENTIFIED"].astype(str).str.strip().str.upper().isin(_UNIDENT_TRUE)
            chunk = chunk[~unident]
            # ── drop rows whose series-key geography/code is missing or sentinel ──
            for c in keys:
                code = chunk[c].astype(str).str.strip().str.upper()
                chunk = chunk[chunk[c].notna() & ~code.isin(_UNIDENT_SENTINELS)]
            if chap is not None:
                chunk = chunk[chunk["BNF_CHAPTER_PLUS_CODE"] == chap]
            if chunk.empty:
                continue
            # ── harmonize geography codes across the July-2022 reorg (identity if None) ──
            if crosswalk:
                for col, mapping in crosswalk.items():
                    if col in chunk.columns:
                        chunk[col] = chunk[col].map(lambda v: mapping.get(v, v))
            chunk["ym"] = chunk["YEAR_MONTH"].map(parse_year_month)
            if ym_lo is not None: chunk = chunk[chunk["ym"] >= ym_lo]
            if ym_hi is not None: chunk = chunk[chunk["ym"] <= ym_hi]
            if chunk.empty:
                continue
            # sum across SNOMED/presentation multiplicity up to the series key
            agg = chunk.groupby(gb_cols, observed=True)[AGG_COLS].sum().reset_index()
            parts.append(agg)

    if not parts:
        raise RuntimeError(
            "No usable rows after filtering. Check that your files post-date the "
            "July-2022 ICB reorg, or supply cfg['geo_crosswalk'] to map legacy "
            "STP/CCG codes onto ICB codes.")
    long = pd.concat(parts, ignore_index=True)
    # collapse duplicates that spanned chunks/files
    long = long.groupby(gb_cols, observed=True)[AGG_COLS].sum().reset_index()
    return _finalize_epd_cache(cfg, long)


def prepare_synthetic_epd_cache(cfg, n_icb=12, n_chem=40, n_months=96, seed=0):
    """Fabricate EPD-shaped monthly demand (trend + 12-month seasonality + noise +
    a price channel) so the full pipeline runs without the raw files. Calendar is
    real (a global month grid), exactly like the true data."""
    rng = np.random.default_rng(seed)
    base_ord = 2016 * 12 + 0                       # global index 0 == Jan 2016
    regions = [f"Y{60+i}" for i in range(4)]
    chapters = ["04: Central Nervous System", "02: Cardiovascular System",
                "05: Infections", "03: Respiratory System"]
    rows = []
    for i in range(n_icb):
        icb = f"ICB{i:02d}"; region = regions[i % len(regions)]
        for j in range(n_chem):
            chem = f"CHEM{j:04d}"; chapter = chapters[j % len(chapters)]
            level = rng.uniform(80, 4000)
            trend = rng.normal(0, level * 0.0015, 1)[0]
            seas_amp = rng.uniform(0.05, 0.30) * level
            phase = rng.uniform(0, 2 * np.pi)
            price = rng.uniform(1.5, 40.0)
            start = int(rng.integers(0, n_months // 4))   # staggered series births
            for t in range(start, n_months):
                ordm = base_ord + t
                moy = ordm % 12
                seas = seas_amp * np.sin(2 * np.pi * moy / 12 + phase)
                mean = max(0.0, level + trend * (t - start) + seas)
                items = max(0.0, rng.normal(mean, mean * 0.08 + 1.0))
                items = float(np.round(items))
                qty = items * rng.uniform(20, 60)
                cost = items * price * rng.uniform(0.9, 1.1)
                qty = items * rng.uniform(20, 60)
                cost = items * price * rng.uniform(0.9, 1.1)
                rows.append((icb, chem, region, chapter, ordm,
                            items, qty, cost * 0.95, cost))
    long = pd.DataFrame(rows, columns=cfg["series_keys"] + STATIC_CAT_COLS[:1] +
                        ["BNF_CHAPTER_PLUS_CODE", "ym"] + AGG_COLS)
    # reorder to canonical column layout
    long = long.rename(columns={cfg["series_keys"][0]: "ICB_CODE",
                                cfg["series_keys"][1]: "BNF_CHEMICAL_SUBSTANCE_CODE"})
    return _finalize_epd_cache(cfg, long)


def _finalize_epd_cache(cfg, long: pd.DataFrame):
    """Pivot the long (series, month) table onto a shared global month grid,
    encode statics, log1p+standardize the target & observed covariates on TRAIN
    months only, compute the temporal split, and write the cache + stats."""
    keys = cfg["series_keys"]
    L, H, stride = cfg["encoder_len"], cfg["horizon"], cfg["stride"]
    T = L + H

    # ---- global calendar grid -----------------------------------------------
    ym_min, ym_max = int(long["ym"].min()), int(long["ym"].max())
    n_months = ym_max - ym_min + 1
    base_year, base_moy = divmod(ym_min, 12)         # calendar of global idx 0
    long["gidx"] = long["ym"] - ym_min               # 0..n_months-1

    # temporal split boundaries (by the calendar month being forecast)
    test_start = n_months - cfg["test_months"]
    val_start = test_start - cfg["val_months"]
    assert val_start - (L + H) > 0, "Not enough history before the val split."

    # ---- one row per series with its static descriptors ---------------------
    series_meta = (long.sort_values("gidx")
                   .groupby(keys, observed=True)
                   .agg(first_g=("gidx", "min"), last_g=("gidx", "max"),
                        n_active=("gidx", "nunique"))
                   .reset_index())
    series_meta = series_meta[series_meta["n_active"] >= cfg["min_active_months"]]

    # rank by volume; optionally keep only the top-K busiest series
    vol = long.groupby(keys, observed=True)["ITEMS"].sum().rename("vol")
    series_meta = series_meta.merge(vol, on=keys, how="left")
    series_meta = series_meta.sort_values("vol", ascending=False)
    if cfg.get("top_k_series"):
        series_meta = series_meta.head(int(cfg["top_k_series"]))
    series_meta = series_meta.reset_index(drop=True)
    series_meta["series_id"] = np.arange(len(series_meta))
    keep = set(map(tuple, series_meta[keys].itertuples(index=False, name=None)))

    # ---- dense per-series channel arrays on the global grid -----------------
    long_k = long.set_index(keys)
    key_to_sid = {tuple(r[keys].values): int(r["series_id"])
                  for _, r in series_meta.iterrows()}

    # accumulate per (series, channel) on the full month grid
    raw_series: Dict[int, np.ndarray] = {
        sid: np.full((n_months, len(AGG_COLS)), np.nan, np.float64)
        for sid in series_meta["series_id"]}
    active_span: Dict[int, Tuple[int, int]] = {}

    gsub = long.groupby(keys, observed=True)
    for kv, g in gsub:
        kv = kv if isinstance(kv, tuple) else (kv,)
        if kv not in keep:
            continue
        sid = key_to_sid[kv]
        gi = g["gidx"].to_numpy()
        raw_series[sid][gi] = g[AGG_COLS].to_numpy(np.float64)
        active_span[sid] = (int(gi.min()), int(gi.max()))

    # ---- derive channels: target=log1p(items); per-item cost/qty/adq --------
    # train mask = months strictly before the val split (no leakage in stats)
    train_g = np.arange(n_months) < val_start

    def per_item(num, den):
        with np.errstate(invalid="ignore", divide="ignore"):
            r = num / den
        return r

    obs_train_pool = {c: [] for c in ["logdemand", "cost_pi", "qty_pi"]}
    derived: Dict[int, np.ndarray] = {}
    static_scale = {}
    for sid, arr in raw_series.items():
        items = arr[:, 0]
        # demand: a NaN month inside the active span is genuine zero demand
        a0, a1 = active_span.get(sid, (0, n_months - 1))
        span = np.zeros(n_months, bool); span[a0:a1 + 1] = True
        items_filled = np.where(span & np.isnan(items), 0.0, items)
        logdemand = np.log1p(np.where(np.isnan(items_filled), 0.0, items_filled))

        cost_pi = per_item(arr[:, 3], items)         # ACTUAL_COST / ITEMS
        qty_pi = per_item(arr[:, 1], items)          # TOTAL_QUANTITY / ITEMS

        # fill undefined per-item channels within span (ffill/bfill/series median)
        def fill(x):
            s = pd.Series(x).where(span)
            s = s.ffill().bfill()
            s = s.fillna(s.median() if np.isfinite(s.median()) else 0.0).fillna(0.0)
            return s.to_numpy(np.float64)
        cost_pi, qty_pi = fill(cost_pi), fill(qty_pi)

        d = np.stack([logdemand, cost_pi, qty_pi], axis=1)  # (n_months, 3)
        derived[sid] = d

        # static scale features: series log-mean & log-std demand (train span)
        tr = logdemand[train_g & span]
        tr = tr if tr.size else logdemand[span]
        static_scale[sid] = (float(np.mean(tr)) if tr.size else 0.0,
                             float(np.std(tr)) if tr.size else 0.0)

        m = train_g & span
        if m.any():
            obs_train_pool["logdemand"].append(logdemand[m])
            obs_train_pool["cost_pi"].append(cost_pi[m])
            obs_train_pool["qty_pi"].append(qty_pi[m])


    # ---- standardization stats (TRAIN months only) --------------------------
    def mean_std(chunks):
        x = np.concatenate(chunks) if chunks else np.zeros(1)
        mu, sd = float(np.mean(x)), float(np.std(x))
        return mu, (sd if sd > 1e-6 else 1.0)

    obs_names = ["logdemand", "cost_pi", "qty_pi"]
    obs_mean, obs_std = {}, {}
    for nm in obs_names:
        obs_mean[nm], obs_std[nm] = mean_std(obs_train_pool[nm])
    t_mean, t_std = obs_mean["logdemand"], obs_std["logdemand"]   # target == channel 0

    # static-real (series scale) stats over train
    sc = np.array(list(static_scale.values()), float) if static_scale else np.zeros((1, 2))
    sc_mean, sc_std = sc.mean(0), np.where(sc.std(0) < 1e-6, 1.0, sc.std(0))

    # ---- standardize & write per-series observed arrays ---------------------
    os.makedirs(os.path.join(cfg["cache_dir"], "series"), exist_ok=True)
    means = np.array([obs_mean[n] for n in obs_names])
    stds = np.array([obs_std[n] for n in obs_names])
    for sid, d in derived.items():
        z = (d - means) / stds
        np.save(os.path.join(cfg["cache_dir"], "series", f"{sid}.npy"),
                z.astype(np.float32))

    # ---- encode static categoricals -----------------------------------------
    cardinalities, maps = [], {}
    for c in STATIC_CAT_COLS:
        vocab = sorted(series_meta[c].fillna("UNK").astype(str).unique().tolist())
        m = {v: i for i, v in enumerate(vocab)}
        series_meta[c + "_id"] = series_meta[c].fillna("UNK").astype(str).map(m).astype(int)
        cardinalities.append(len(vocab)); maps[c] = m

    # static reals (standardized)
    sm = series_meta["series_id"].map(lambda s: static_scale[s][0]).to_numpy(float)
    sv = series_meta["series_id"].map(lambda s: static_scale[s][1]).to_numpy(float)
    series_meta["scale_mean_z"] = (sm - sc_mean[0]) / sc_std[0]
    series_meta["scale_std_z"] = (sv - sc_mean[1]) / sc_std[1]
    series_meta["active_start"] = series_meta["series_id"].map(lambda s: active_span[s][0])
    series_meta["active_end"] = series_meta["series_id"].map(lambda s: active_span[s][1])

    stats = {
        "series_keys": keys, "target_col": cfg["target_col"], "log_transform": True,
        "t_mean": t_mean, "t_std": t_std,
        "obs_names": obs_names, "obs_mean": obs_mean, "obs_std": obs_std,
        "static_cat_cols": STATIC_CAT_COLS, "static_cat_cardinalities": cardinalities,
        "static_cat_maps": maps,
        "scale_mean": sc_mean.tolist(), "scale_std": sc_std.tolist(),
        "known_cat_cardinalities": [12],          # month-of-year
        "n_known_real": 3,                         # sin, cos, trend
        "n_static_real": 2,                        # series log-mean & log-std
        "n_observed_real": len(obs_names),
        "target_idx": 0,
        "encoder_len": L, "horizon": H, "stride": stride,
        "n_months": int(n_months), "base_year": int(base_year), "base_moy": int(base_moy),
        "val_start": int(val_start), "test_start": int(test_start),
    }
    _save_epd_cache(cfg["cache_dir"], cfg["splits_dir"], series_meta, stats)
    return stats


def _save_epd_cache(cache_dir, splits_dir, series_meta, stats):
    os.makedirs(cache_dir, exist_ok=True); os.makedirs(splits_dir, exist_ok=True)
    series_meta.to_parquet(os.path.join(cache_dir, "static.parquet"))
    with open(os.path.join(cache_dir, "feature_stats.json"), "w") as f:
        json.dump(stats, f, indent=2)

    L, H, stride = stats["encoder_len"], stats["horizon"], stats["stride"]
    T = L + H
    val_start, test_start = stats["val_start"], stats["test_start"]

    def split_of(last_g):                 # by the calendar month of the LAST horizon step
        if last_g < val_start:  return "train"
        if last_g < test_start: return "val"
        return "test"

    rows = {"train": [], "val": [], "test": []}
    for _, r in series_meta.iterrows():
        sid = int(r["series_id"]); a0, a1 = int(r["active_start"]), int(r["active_end"])
        # only windows fully inside the series' active calendar span
        for t0 in range(a0, a1 - T + 2, stride):
            if t0 < 0 or t0 + T - 1 > a1:
                continue
            rows[split_of(t0 + T - 1)].append((sid, t0))
    for split, rws in rows.items():
        pd.DataFrame(rws, columns=["series_id", "t0"]).to_csv(
            os.path.join(splits_dir, f"windows_{split}.csv"), index=False)
    n = {k: len(v) for k, v in rows.items()}
    print(f"[cache] {len(series_meta)} series · {stats['n_months']} months · "
          f"windows train/val/test = {n['train']}/{n['val']}/{n['test']}")

In [ ]:
# =============================================================================
# Dataset
# =============================================================================
class EPDForecastDataset(Dataset):
    """One (encoder, decoder, target) window per index, in the exact key layout
    ModernTFT.forward expects. Series arrays are pre-standardized in the cache and
    indexed by the GLOBAL month grid, so known-future calendar features are real."""

    def __init__(self, cache_dir, splits_dir, split, stats):
        self.cache_dir = cache_dir; self.stats = stats
        self.L = stats["encoder_len"]; self.H = stats["horizon"]; self.T = self.L + self.H
        self.target_idx = stats["target_idx"]
        self.base_moy = stats["base_moy"]; self.n_months = stats["n_months"]

        self.windows = pd.read_csv(os.path.join(splits_dir, f"windows_{split}.csv"))
        st = pd.read_parquet(os.path.join(cache_dir, "static.parquet")).set_index("series_id")
        self.cat_cols = [c + "_id" for c in stats["static_cat_cols"]]
        self.static = st
        self._series_cache: Dict[int, np.ndarray] = {}

    def __len__(self):
        return len(self.windows)

    def _series(self, sid):
        a = self._series_cache.get(sid)
        if a is None:
            a = np.load(os.path.join(self.cache_dir, "series", f"{sid}.npy"))
            self._series_cache[sid] = a
        return a

    def __getitem__(self, i):
        sid = int(self.windows.iloc[i]["series_id"]); t0 = int(self.windows.iloc[i]["t0"])
        s = self._series(sid)
        L, H, T = self.L, self.H, self.T
        enc = s[t0:t0 + L]                                # (L, n_obs)
        target = s[t0 + L:t0 + T, self.target_idx]        # (H,)  z-scored log-demand

        row = self.static.loc[sid]
        static_cat = np.array([row[c] for c in self.cat_cols], dtype=np.int64)
        static_real = np.array([row["scale_mean_z"], row["scale_std_z"]], dtype=np.float32)

        # known-future calendar features from the absolute global month index
        g = t0 + np.arange(T)                             # (T,) global month indices
        moy = (self.base_moy + g) % 12                    # month-of-year 0..11
        ang = 2 * np.pi * moy / 12.0
        trend = (g / max(1, self.n_months - 1)).astype(np.float32)
        known_real = np.stack([np.sin(ang), np.cos(ang), trend], axis=-1).astype(np.float32)

        return {
            "static_cat":    torch.from_numpy(static_cat),
            "static_real":   torch.from_numpy(static_real),
            "known_cat":     torch.from_numpy(moy.astype(np.int64))[:, None],   # (T,1)
            "known_real":    torch.from_numpy(known_real),                      # (T,3)
            "observed_real": torch.from_numpy(enc.astype(np.float32)),          # (L,n_obs)
            "target":        torch.from_numpy(target.astype(np.float32)),       # (H,)
            "series_id":     sid,
            "t0":            t0,
            "horizon_start_g": t0 + L,
            "sample_name":   f"{sid}_{t0}",
        }


def make_datasets(cfg, stats) -> Dict[str, Dataset]:
    out = {}
    for split in ("train", "val", "test"):
        wpath = os.path.join(cfg["splits_dir"], f"windows_{split}.csv")
        if os.path.exists(wpath) and len(pd.read_csv(wpath)):
            out[split] = EPDForecastDataset(cfg["cache_dir"], cfg["splits_dir"], split, stats)
    return out


def _make_loaders(datasets, batch_size, num_workers, cfg):
    """Returns (loaders, None). None kept for API parity with the BirdCLEF
    trainer (curriculum/domain-sampler hook); unused here."""
    loaders = {}
    for split, ds in datasets.items():
        loaders[split] = DataLoader(
            ds, batch_size=batch_size, shuffle=(split == "train"),
            num_workers=num_workers, pin_memory=(cfg["device"] != "cpu"),
            drop_last=(split == "train"), persistent_workers=(num_workers > 0),
        )
    return loaders, None


# =============================================================================
# Loss  (pinball + ramped non-crossing penalty)  -- unchanged from MIMIC
# =============================================================================
class CombinedQuantileLoss(nn.Module):
    def __init__(self, quantiles, aux_weight=0.0):
        super().__init__()
        self.register_buffer("q", torch.tensor(quantiles, dtype=torch.float32))
        self.aux_weight = float(aux_weight)
    def set_aux_weight(self, w: float):
        self.aux_weight = float(w)
    def forward(self, preds, target):
        e = target.unsqueeze(-1) - preds
        pinball = torch.maximum(self.q * e, (self.q - 1.0) * e).mean()
        cross = F.relu(preds[..., :-1] - preds[..., 1:]).mean()      # non-crossing
        return pinball + self.aux_weight * cross


# =============================================================================
# Forecast metrics (numpy). Point metrics & sharpness in ITEM units (expm1);
# coverage & pinball are computed in standardized space (monotonic-invariant).
# Adds WAPE, the standard demand/stock forecasting error.
# =============================================================================
def forecast_metrics(preds, target, quantiles, inv=None):
    q = np.asarray(quantiles)
    med_i = int(np.argmin(np.abs(q - 0.5)))
    lo_i, hi_i = int(np.argmin(q)), int(np.argmax(q))

    cov = float(np.mean((target >= preds[..., lo_i]) & (target <= preds[..., hi_i])))
    eq = target[..., None] - preds
    pinball = float(np.mean(np.maximum(q * eq, (q - 1.0) * eq)))

    if inv is None:
        inv = lambda z: z
    med_u = inv(preds[..., med_i]); tgt_u = inv(target)
    lo_u, hi_u = inv(preds[..., lo_i]), inv(preds[..., hi_i])
    e = tgt_u - med_u
    mae = float(np.mean(np.abs(e)))
    rmse = float(np.sqrt(np.mean(e ** 2)))
    sharp = float(np.mean(hi_u - lo_u))
    wape = float(np.sum(np.abs(e)) / (np.sum(np.abs(tgt_u)) + 1e-8))
    return {"qloss": pinball, "mae": mae, "rmse": rmse,
            "coverage": cov, "sharpness": sharp, "wape": wape}

In [ ]:
# =============================================================================
# Trainer
# =============================================================================
class EPDForecastTrainer:

    def __init__(self, config: dict):
        self.cfg = config
        torch.manual_seed(config.get("seed", 0)); np.random.seed(config.get("seed", 0))
        self._setup_dirs(); self._setup_logging()
        self.is_train = self.cfg["is_train"]

        print(f"\n{'='*70}\nNHS EPD · ModernTFT Demand Forecaster\n{'='*70}\n")

        with open(os.path.join(config["cache_dir"], "feature_stats.json")) as f:
            self.stats = json.load(f)
        self.quantiles = tuple(config["quantiles"])
        self.target_idx = self.stats["target_idx"]
        self.t_mean = self.stats["t_mean"]; self.t_std = self.stats["t_std"]
        # inverse transform: standardized log-demand -> item counts
        self.inv = lambda z: np.expm1(z * self.t_std + self.t_mean)

        print("Building datasets...")
        self.datasets = make_datasets(config, self.stats)
        self.loaders, self.domain_sampler = _make_loaders(
            self.datasets, batch_size=config["batch_size"],
            num_workers=config.get("num_workers", 4), cfg=config)
        for k, ld in self.loaders.items():
            print(f"  {k:5s}: {len(ld.dataset):>8,} windows  ({len(ld)} batches)")

        print("Building model...")
        tcfg = TFTConfig(
            static_cat_cardinalities=tuple(self.stats["static_cat_cardinalities"]),
            n_static_real=self.stats["n_static_real"],
            known_cat_cardinalities=tuple(self.stats["known_cat_cardinalities"]),
            n_known_real=self.stats["n_known_real"],
            n_observed_real=self.stats["n_observed_real"],
            d_model=config["d_model"], n_heads=config["n_heads"],
            n_kv_heads=config["n_kv_heads"], n_blocks=config["n_blocks"],
            dropout=config["dropout"], quantiles=self.quantiles,
            max_len=config["encoder_len"] + config["horizon"] + 8,
        )
        self.tcfg = tcfg
        self.model = ModernTFT(tcfg).to(config["device"])
        n_params = sum(p.numel() for p in self.model.parameters())
        print(f"  ModernTFT: {n_params:,} params | d_model={tcfg.d_model} "
              f"d_ff={tcfg.d_ff} blocks={tcfg.n_blocks} kv_heads={tcfg.n_kv_heads} "
              f"| static_card={self.stats['static_cat_cardinalities']}")

        self.criterion = CombinedQuantileLoss(self.quantiles, aux_weight=0.0).to(config["device"])

        # optimiser: slow feature stack / fast attention+head (unchanged idiom)
        SLOW = ("emb", "lstm", "vsn_", "grn_", "enrich")
        slow, fast = [], []
        for n, p in self.model.named_parameters():
            if not p.requires_grad: continue
            (slow if any(s in n for s in SLOW) else fast).append(p)
        self.optimizer = torch.optim.AdamW([
            {"params": slow, "lr": config["lr"] * config["feature_lr_mult"]},
            {"params": fast, "lr": config["lr"]},
        ], weight_decay=config["weight_decay"], betas=(0.9, 0.95))

        warmup = config.get("warmup", 5); restart_epoch = config.get("restart_epoch", 50)
        self.scheduler = torch.optim.lr_scheduler.SequentialLR(
            self.optimizer,
            schedulers=[
                torch.optim.lr_scheduler.LinearLR(
                    self.optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup),
                torch.optim.lr_scheduler.CosineAnnealingLR(
                    self.optimizer, eta_min=1e-6, T_max=max(1, restart_epoch - warmup)),
            ], milestones=[warmup])
        self._restart_epoch = restart_epoch
        self._restart_lr = config.get("restart_lr", 3e-5)
        self._restart_epochs_remaining = config["num_epochs"] - restart_epoch
        self._phase2_scheduler = None

        self.scaler = (torch.amp.GradScaler("cuda")
                       if config.get("use_amp", True) and config["device"] != "cpu" else None)

        self.current_epoch = 0
        self.best_val = math.inf
        self.early_stopping = EarlyStopping(
            patience=config.get("early_stopping_patience", 15),
            min_delta=config.get("early_stopping_min_delta", 1e-4))
        self.history = self._load_history()

        resume = config.get("resume", "")
        if resume and Path(resume).exists():
            self._load_checkpoint(resume)

    # ------------------------------------------------------------------ #
    def _setup_dirs(self):
        cfg = self.cfg
        for d in [cfg["logdir"], os.path.join(cfg["logdir"], "figures"),
                  str(Path(cfg.get("resume", os.path.join(cfg["logdir"], "best.pth"))).parent)]:
            os.makedirs(d, exist_ok=True)

    def _setup_logging(self):
        logdir = self.cfg["logdir"]
        # Anchor tees to the REAL stdout. __stdout__ is the original console even
        # if an outer trainer already redirected sys.stdout (test-mode creates a
        # second trainer), so we never nest tees and double-write.
        base = sys.__stdout__ or sys.stdout
        self._train_tee = TeeFile(base, os.path.join(logdir, "train.log"))
        self._val_tee = TeeFile(base, os.path.join(logdir, "val.log"))

    def _log_stdout(self):
        """Context manager: route every print() in the block into train.log too.
        Without this, bare print()s (epoch summaries, 'Best model saved', the
        banners) go only to the console and the log file stays empty."""
        from contextlib import redirect_stdout
        return redirect_stdout(self._train_tee)

    def _load_history(self) -> dict:
        path = os.path.join(self.cfg["logdir"], "history.json")
        if Path(path).exists():
            with open(path) as f: return json.load(f)
        return {"train_loss": [], "train_mae": [], "val_loss": [], "val_mae": [],
                "val_rmse": [], "val_wape": [], "val_coverage": [], "val_sharpness": [], "lr": []}

    def _save_history(self):
        with open(os.path.join(self.cfg["logdir"], "history.json"), "w") as f:
            json.dump(self.history, f, indent=2)

    @staticmethod
    def _ramp(epoch, start_epoch, end_epoch, max_value):
        if epoch < start_epoch: return 0.0
        if epoch >= end_epoch:  return max_value
        return float(max_value * (epoch - start_epoch) / max(1, end_epoch - start_epoch))

    def _save_checkpoint(self, epoch, val_loss, is_best):
        ckpt = {"model": self.model.state_dict(), "optimizer": self.optimizer.state_dict(),
                "scheduler": self.scheduler.state_dict(), "epoch": epoch,
                "best_val": self.best_val, "tft_config": asdict(self.tcfg),
                "stats": self.stats,
                "config": {k: v for k, v in self.cfg.items() if k != "device"}}
        torch.save(ckpt, os.path.join(self.cfg["logdir"], "latest.pth"))
        if is_best:
            best_path = self.cfg.get("resume", os.path.join(self.cfg["logdir"], "best.pth"))
            torch.save(ckpt, best_path)
            print(f"  ✓ Best model saved (val qloss {val_loss:.4f}) → {best_path}")

    def _load_checkpoint(self, path):
        print(f"Resuming from {path}")
        ckpt = torch.load(path, map_location=self.cfg["device"], weights_only=False)
        self.model.load_state_dict(ckpt["model"], strict=False)
        if "optimizer" in ckpt: self.optimizer.load_state_dict(ckpt["optimizer"])
        if "scheduler" in ckpt: self.scheduler.load_state_dict(ckpt["scheduler"])
        if "epoch" in ckpt: self.current_epoch = ckpt["epoch"] + 1
        if "best_val" in ckpt: self.best_val = ckpt["best_val"]
        print(f"  Resumed: epoch {self.current_epoch}, best val {self.best_val:.4f}")

    def _to_device(self, batch):
        return {k: (v.to(self.cfg["device"], non_blocking=True) if torch.is_tensor(v) else v)
                for k, v in batch.items()}

    # ------------------------------------------------------------------ #
    def _train_epoch(self, epoch) -> dict:
        self.current_epoch = epoch
        self.model.train()
        loader = self.loaders["train"]; total = len(loader)
        accum = self.cfg.get("accumulation", 1); clip = self.cfg.get("grad_clip", 1.0)
        metric_stride = self.cfg.get("train_metric_stride", 4)

        running_loss = 0.0; all_preds, all_targets = [], []
        self.optimizer.zero_grad(set_to_none=True)
        looper = tqdm(enumerate(loader), total=total, desc=f"Train {epoch+1}", file=self._train_tee)

        for step, batch in looper:
            batch = self._to_device(batch); target = batch["target"]

            if (step % metric_stride) == 0:
                self.model.eval()
                with torch.no_grad():
                    clean = self.model(batch)["prediction"].float().cpu().numpy()
                all_preds.append(clean); all_targets.append(target.detach().cpu().numpy())
                self.model.train()

            if self.scaler is not None:
                with torch.amp.autocast("cuda"):
                    preds = self.model(batch)["prediction"]; loss = self.criterion(preds, target)
                self.scaler.scale(loss / accum).backward()
                if (step + 1) % accum == 0 or step == total - 1:
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=clip)
                    self.scaler.step(self.optimizer); self.scaler.update()
                    self.optimizer.zero_grad(set_to_none=True)
            else:
                preds = self.model(batch)["prediction"]; loss = self.criterion(preds, target)
                (loss / accum).backward()
                if (step + 1) % accum == 0 or step == total - 1:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=clip)
                    self.optimizer.step(); self.optimizer.zero_grad(set_to_none=True)

            running_loss += loss.item()
            looper.set_postfix({"loss": f"{running_loss/(step+1):.4f}",
                                "lr": f"{self.optimizer.param_groups[1]['lr']:.1e}"})

        if all_preds:
            P = np.concatenate(all_preds, 0); Tg = np.concatenate(all_targets, 0)
            m = forecast_metrics(P, Tg, self.quantiles, self.inv); mae = m["mae"]
        else:
            mae = float("nan")
        return {"loss": running_loss / total, "mae": mae}

    @torch.no_grad()
    def audit_feature_statistics(self):
        for split in ("train", "val"):
            if split not in self.loaders: continue
            loader = self.loaders[split]; rows = []
            for i, batch in enumerate(loader):
                if i >= 20: break
                obs = batch["observed_real"].float()
                rows.append((obs.mean().item(), obs.std().item(),
                             batch["target"].float().mean().item()))
            a = np.array(rows)
            print(f"  {split}: obs mean={a[:,0].mean():+.3f} std={a[:,1].mean():.3f} "
                  f"target(z) mean={a[:,2].mean():+.3f}")

    # ------------------------------------------------------------------ #
    def _validate(self, epoch) -> dict:
        self.model.eval()
        loader = self.loaders["val"]; running_loss = 0.0
        all_preds, all_targets = [], []
        looper = tqdm(enumerate(loader), total=len(loader), desc=f"Val   {epoch+1}", file=self._val_tee)
        with torch.no_grad():
            for step, batch in looper:
                batch = self._to_device(batch); target = batch["target"]
                if self.scaler is not None:
                    with torch.amp.autocast("cuda"):
                        preds = self.model(batch)["prediction"]; loss = self.criterion(preds, target)
                else:
                    preds = self.model(batch)["prediction"]; loss = self.criterion(preds, target)
                running_loss += loss.item()
                all_preds.append(preds.float().cpu().numpy())
                all_targets.append(target.float().cpu().numpy())
                looper.set_postfix({"loss": f"{running_loss/(step+1):.4f}"})
        P = np.concatenate(all_preds, 0); Tg = np.concatenate(all_targets, 0)
        m = forecast_metrics(P, Tg, self.quantiles, self.inv)
        m["loss"] = running_loss / len(loader); m["preds"] = P; m["targets"] = Tg
        return m

    # ------------------------------------------------------------------ #
    def train(self):
        # mirror everything printed during training into train.log
        with self._log_stdout():
            return self._train_impl()

    def _train_impl(self):
        print(f"\n{'='*70}")
        print(f"Training  |  target={self.cfg['target_col']}  |  "
              f"L={self.cfg['encoder_len']} H={self.cfg['horizon']} months  |  "
              f"epochs {self.current_epoch}→{self.cfg['num_epochs']}")
        print(f"{'='*70}\n")

        aux_max = float(self.cfg.get("aux_weight_max", 0.1))
        aux_s = int(self.cfg.get("aux_ramp_start_epoch", 5))
        aux_e = int(self.cfg.get("aux_ramp_end_epoch", 20))
        self.audit_feature_statistics()

        # ---- training-time accounting (for the TFT-vs-FM cost comparison) ----
        # Foundation models are zero-shot (no training); the TFT pays this cost
        # once. We record total wall-clock, per-epoch seconds, device, and param
        # count so the comparison can report accuracy *and* training effort.
        train_t_start = time.time()
        epoch_seconds: List[float] = []
        n_params = sum(p.numel() for p in self.model.parameters())

        for epoch in range(self.current_epoch, self.cfg["num_epochs"]):
            t0 = time.time()
            aux_w = self._ramp(epoch, aux_s, aux_e, aux_max)
            self.criterion.set_aux_weight(aux_w)
            print(f"  [loss] non-crossing aux_w={aux_w:.3f}")

            train_m = self._train_epoch(epoch)
            val_m = self._validate(epoch)

            if epoch + 1 == self._restart_epoch and self._restart_epochs_remaining > 0:
                print(f"\n  >>> WARM RESTART: LR → {self._restart_lr:.1e}, "
                      f"{self._restart_epochs_remaining} epochs remaining")
                top = max(g["lr"] for g in self.optimizer.param_groups)
                for pg in self.optimizer.param_groups:
                    pg["lr"] = self._restart_lr * max(pg["lr"] / top, 0.1)
                self._phase2_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    self.optimizer, eta_min=1e-6, T_max=self._restart_epochs_remaining)

            if self._phase2_scheduler is not None and epoch >= self._restart_epoch:
                self._phase2_scheduler.step()
            else:
                self.scheduler.step()

            lr = self.optimizer.param_groups[1]["lr"]
            print(
                f"\nEpoch {epoch+1:3d}/{self.cfg['num_epochs']}  ({time.time()-t0:.0f}s)\n"
                f"  Train | loss {train_m['loss']:.4f}  MAE {train_m['mae']:.1f} items\n"
                f"  Val   | loss {val_m['loss']:.4f}  MAE {val_m['mae']:.1f} items  "
                f"RMSE {val_m['rmse']:.1f}  WAPE {val_m['wape']:.3f}  "
                f"cover[{self.quantiles[0]:.0%}-{self.quantiles[-1]:.0%}] {val_m['coverage']:.3f}  "
                f"sharp {val_m['sharpness']:.1f}  lr {lr:.1e}")

            self.history["train_loss"].append(float(train_m["loss"]))
            self.history["train_mae"].append(float(train_m["mae"]))
            self.history["val_loss"].append(float(val_m["loss"]))
            self.history["val_mae"].append(float(val_m["mae"]))
            self.history["val_rmse"].append(float(val_m["rmse"]))
            self.history["val_wape"].append(float(val_m["wape"]))
            self.history["val_coverage"].append(float(val_m["coverage"]))
            self.history["val_sharpness"].append(float(val_m["sharpness"]))
            self.history["lr"].append(float(lr))
            self._save_history()

            is_best = val_m["loss"] < self.best_val
            if is_best: self.best_val = val_m["loss"]
            self._save_checkpoint(epoch, val_m["loss"], is_best)

            if (epoch + 1) % 5 == 0:
                try:
                    self._plot_metrics(epoch + 1)
                    self._plot_forecast_examples(val_m["preds"], val_m["targets"], epoch + 1)
                    self._plot_calibration(val_m["preds"], val_m["targets"], epoch + 1)
                except Exception as e:
                    print(f"  [plot] skipped ({e})")

            epoch_seconds.append(time.time() - t0)

            if self.early_stopping.step(val_m["loss"]):
                print(f"  Early stopping at epoch {epoch+1} (no improvement)."); break

        # ---- persist the training-time record (consumed by the FM benchmark) ----
        total_train_s = time.time() - train_t_start
        n_epochs_run = len(epoch_seconds)
        n_train_windows = len(self.loaders["train"].dataset) if "train" in self.loaders else 0
        timing = {
            "device": str(self.cfg["device"]),
            "n_params": int(n_params),
            "epochs_run": int(n_epochs_run),
            "epochs_configured": int(self.cfg["num_epochs"]),
            "train_windows": int(n_train_windows),
            "total_train_seconds": float(total_train_s),
            "total_train_minutes": float(total_train_s / 60.0),
            "mean_epoch_seconds": float(np.mean(epoch_seconds)) if epoch_seconds else 0.0,
            "median_epoch_seconds": float(np.median(epoch_seconds)) if epoch_seconds else 0.0,
            "epoch_seconds": [float(s) for s in epoch_seconds],
            "best_val_qloss": float(self.best_val),
        }
        with open(os.path.join(self.cfg["logdir"], "train_timing.json"), "w") as f:
            json.dump(timing, f, indent=2)
        self.history["train_timing"] = timing
        self._save_history()

        print(f"\n{'='*70}\nTraining complete.  Best val qloss: {self.best_val:.4f}")
        print(f"  Training cost: {total_train_s/60.0:.2f} min on {self.cfg['device']} "
              f"({n_epochs_run} epochs, {timing['mean_epoch_seconds']:.1f}s/epoch, "
              f"{n_params:,} params)")
        print(f"  → {os.path.join(self.cfg['logdir'], 'train_timing.json')}")
        print(f"  NOTE: foundation models below are ZERO-SHOT (training cost = 0).\n{'='*70}\n")

    # ------------------------------------------------------------------ #
    def predict_test(self, model_path: Optional[str] = None) -> pd.DataFrame:
        with self._log_stdout():
            return self._predict_test_impl(model_path)

    def _predict_test_impl(self, model_path: Optional[str] = None) -> pd.DataFrame:
        if "test" not in self.loaders:
            raise RuntimeError("No test split found in cache.")
        if model_path is None:
            model_path = self.cfg.get("resume", os.path.join(self.cfg["logdir"], "best.pth"))
        if Path(model_path).exists():
            print(f"\nLoading best model: {model_path}")
            ckpt = torch.load(model_path, map_location=self.cfg["device"], weights_only=False)
            self.model.load_state_dict(ckpt["model"], strict=False)
        self.model.eval()

        base_year, base_moy = self.stats["base_year"], self.stats["base_moy"]
        base_ord = base_year * 12 + base_moy
        qcols = [f"p{int(q*100)}" for q in self.quantiles]
        rows = []
        with torch.no_grad():
            for batch in tqdm(self.loaders["test"], desc="Inference"):
                sid = batch["series_id"]; hstart = batch["horizon_start_g"]
                target = batch["target"].numpy()
                batch = self._to_device(batch)
                if self.scaler is not None:
                    with torch.amp.autocast("cuda"):
                        preds = self.model(batch)["prediction"]
                else:
                    preds = self.model(batch)["prediction"]
                preds = preds.float().cpu().numpy()                 # (B,H,Q) z-space
                preds_u = self.inv(preds)                           # -> item counts
                tgt_u = self.inv(target)
                B, H, Q = preds_u.shape
                for b in range(B):
                    for h in range(H):
                        ym = ord_to_label(base_ord + int(hstart[b]) + h)
                        rows.append([f"{int(sid[b])}_{ym}", int(sid[b]), ym, h + 1,
                                     *preds_u[b, h].tolist(), float(tgt_u[b, h])])

        sub = pd.DataFrame(rows, columns=["row_id", "series_id", "year_month", "step",
                                          *qcols, "target"])
        out_csv = self.cfg.get("submission_csv", "forecasts.csv")
        os.makedirs(os.path.dirname(out_csv) or ".", exist_ok=True)
        sub.to_csv(out_csv, index=False)
        print(f"Forecasts saved → {out_csv}  ({len(sub):,} rows)")
        return sub

    # ------------------------------------------------------------------ #
    # Plots (item-unit scaling via expm1)
    # ------------------------------------------------------------------ #
    def _plot_metrics(self, epoch):
        if not HAS_MPL: return
        save_dir = os.path.join(self.cfg["logdir"], "figures", f"epoch_{epoch}")
        os.makedirs(save_dir, exist_ok=True)
        h = self.history; eps = list(range(1, len(h["train_loss"]) + 1))
        panels = [("loss", "train_loss", "val_loss", "Quantile loss", False),
                  ("mae", "train_mae", "val_mae", "MAE (items)", False),
                  ("wape", None, "val_wape", "WAPE", False),
                  ("coverage", None, "val_coverage", "Interval coverage", False),
                  ("sharpness", None, "val_sharpness", "Sharpness (items)", False),
                  ("lr", None, "lr", "Learning rate", True)]
        for fname, tkey, vkey, title, log_y in panels:
            fig, ax = plt.subplots(figsize=(9, 5))
            if tkey and tkey in h: ax.plot(eps, h[tkey], label="Train", linewidth=2)
            if vkey and vkey in h: ax.plot(eps, h[vkey], label="Val", linewidth=2)
            if fname == "coverage":
                ax.axhline(self.quantiles[-1] - self.quantiles[0], color="red",
                           ls="--", alpha=.7, label="nominal")
            ax.set_xlabel("Epoch"); ax.set_ylabel(title); ax.set_title(title, fontweight="bold")
            if log_y: ax.set_yscale("log")
            ax.legend(); ax.grid(True, ls="--", alpha=.5); fig.tight_layout()
            fig.savefig(os.path.join(save_dir, f"{fname}.png"), dpi=130); plt.close(fig)

    def _plot_forecast_examples(self, preds, targets, epoch, n=6):
        if not HAS_MPL: return
        save_dir = os.path.join(self.cfg["logdir"], "figures", f"epoch_{epoch}")
        os.makedirs(save_dir, exist_ok=True)
        q = np.asarray(self.quantiles)
        med_i = int(np.argmin(np.abs(q - 0.5))); lo_i, hi_i = int(np.argmin(q)), int(np.argmax(q))
        idx = np.linspace(0, len(preds) - 1, min(n, len(preds))).astype(int)
        fig, axes = plt.subplots(2, 3, figsize=(15, 8))
        for ax, i in zip(axes.ravel(), idx):
            hsteps = np.arange(preds.shape[1])
            ax.fill_between(hsteps, self.inv(preds[i, :, lo_i]), self.inv(preds[i, :, hi_i]),
                            color="#5fd0c8", alpha=.25, label="p10–p90")
            ax.plot(hsteps, self.inv(preds[i, :, med_i]), color="#f0b357", lw=2, label="median")
            ax.plot(hsteps, self.inv(targets[i]), color="#222", lw=1.6, ls="--",
                    marker="o", ms=3, label="actual")
            ax.set_xlabel("horizon (months)"); ax.set_ylabel("items")
            ax.grid(True, ls="--", alpha=.4)
        axes.ravel()[0].legend(fontsize=8)
        fig.suptitle(f"Forecast examples (epoch {epoch})", fontweight="bold")
        fig.tight_layout(); fig.savefig(os.path.join(save_dir, "forecast_examples.png"), dpi=130); plt.close(fig)

    def _plot_calibration(self, preds, targets, epoch):
        if not HAS_MPL: return
        save_dir = os.path.join(self.cfg["logdir"], "figures", f"epoch_{epoch}")
        os.makedirs(save_dir, exist_ok=True)
        q = np.asarray(self.quantiles)
        emp = [(targets <= preds[..., k]).mean() for k in range(len(q))]
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.plot([0, 1], [0, 1], color="red", ls="--", alpha=.7, label="ideal")
        ax.plot(q, emp, marker="o", color="#5fd0c8", lw=2, label="empirical")
        ax.set_xlabel("nominal quantile"); ax.set_ylabel("empirical coverage")
        ax.set_title(f"Quantile calibration (epoch {epoch})", fontweight="bold")
        ax.legend(); ax.grid(True, ls="--", alpha=.5); fig.tight_layout()
        fig.savefig(os.path.join(save_dir, "calibration.png"), dpi=130); plt.close(fig)


# =============================================================================
# Entry point
# =============================================================================
build_cache = True
train_mode = True
test_mode = True

if __name__ == "__main__":
    if build_cache:
        if _list_epd_files(CONFIG["epd_raw"]):
            build_epd_cache(CONFIG)
        else:
            print("[cache] raw EPD not found — generating synthetic cache")
            prepare_synthetic_epd_cache(CONFIG)

    if not Path(CONFIG["cache_dir"], "feature_stats.json").exists():
        print("[cache] no cache found — generating synthetic cache")
        prepare_synthetic_epd_cache(CONFIG)

    if train_mode:
        trainer = EPDForecastTrainer(CONFIG)
        trainer.train()

    if test_mode:
        CONFIG["num_workers"] = 0
        trainer = EPDForecastTrainer({**CONFIG, "is_train": False})
        trainer.predict_test()

epd_snomed_202011.csv.gz: 18it [01:30,  5.02s/it]
epd_snomed_202012.csv.gz: 18it [01:18,  4.36s/it]
epd_snomed_202101.csv.gz: 18it [01:14,  4.14s/it]
epd_snomed_202102.csv.gz: 17it [01:13,  4.31s/it]
epd_snomed_202103.csv.gz: 18it [01:20,  4.49s/it]
epd_snomed_202104.csv.gz: 18it [01:15,  4.17s/it]
epd_snomed_202105.csv.gz: 18it [01:15,  4.17s/it]
epd_snomed_202106.csv.gz: 18it [01:18,  4.36s/it]
epd_snomed_202107.csv.gz: 18it [01:17,  4.28s/it]
epd_snomed_202108.csv.gz: 18it [01:14,  4.16s/it]
epd_snomed_202109.csv.gz: 18it [01:17,  4.30s/it]
epd_snomed_202110.csv.gz: 18it [01:15,  4.17s/it]
epd_snomed_202111.csv.gz: 18it [01:17,  4.28s/it]
epd_snomed_202112.csv.gz: 18it [01:18,  4.38s/it]
epd_snomed_202201.csv.gz: 18it [01:15,  4.21s/it]
epd_snomed_202202.csv.gz: 18it [01:14,  4.13s/it]
epd_snomed_202203.csv.gz: 19it [01:20,  4.21s/it]
epd_snomed_202204.csv.gz: 18it [01:15,  4.18s/it]
epd_snomed_202205.csv.gz: 18it [01:22,  4.57s/it]
epd_snomed_202206.csv.gz: 18it [01:22,  4.61s/it]


[cache] 144 series · 65 months · windows train/val/test = 4320/864/1721

NHS EPD · ModernTFT Demand Forecaster

Building datasets...
  train:    4,320 windows  (67 batches)
  val  :      864 windows  (14 batches)
  test :    1,721 windows  (27 batches)
Building model...
  ModernTFT: 627,919 params | d_model=64 d_ff=256 blocks=2 kv_heads=1 | static_card=[7, 21]

NHS EPD · ModernTFT Demand Forecaster

Building datasets...
  train:    4,320 windows  (67 batches)
  val  :      864 windows  (14 batches)
  test :    1,721 windows  (27 batches)
Building model...
  ModernTFT: 627,919 params | d_model=64 d_ff=256 blocks=2 kv_heads=1 | static_card=[7, 21]
Resuming from /content/drive/MyDrive/Dr. Moustafa Saad/mine/TFT/tft_epd_forecast_44/checkpoints/best.pth


Inference:   0%|          | 0/27 [00:00<?, ?it/s]

  Resumed: epoch 27, best val 0.0107


Inference: 100%|██████████| 27/27 [00:02<00:00,  9.40it/s]


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
fm_benchmark.py
===============
Zero-shot foundation-model benchmark on the SAME NHS-EPD test windows your
ModernTFT was evaluated on, so the comparison is apples-to-apples.

Models : Chronos-Bolt, TimesFM 2.5, Moirai (uni2ts), TabPFN-TS, + Seasonal-Naive
Output : one forecast CSV per model (same schema as the TFT's forecasts.csv)
         + a summary table comparing every model vs the target.

WHY IT'S A FAIR COMPARISON
--------------------------
* It reuses the EXACT (series, forecast-origin) windows from the TFT's
  forecasts.csv — same series, same origins, same 6-step horizon, same targets.
* By default each model receives the SAME context the TFT saw: the L-month
  encoder window ending the month before the horizon. (Set CONTEXT_MODE="all"
  to instead give the FMs their full available history — they're built for long
  context, so report both if you want to be generous to them.)
* All forecasts are produced and scored in ITEM units (the model's native target
  was log1p(items); we invert with expm1).

WHAT YOU NEED
-------------
1. The TFT cache dir produced by tft_epd.py  (has feature_stats.json + series/*.npy)
2. The TFT's forecasts.csv  (defines the evaluation windows)
3. A GPU is recommended. TabPFN-TS can run via its cloud client (no GPU).

RUN
---
Edit the CONFIG block, install the model you want (see INSTALL notes per adapter),
then:  python fm_benchmark.py
You can enable/disable models in CONFIG["models"].
"""
!pip uninstall -y torchvision
!pip install chronos-forecasting          # Chronos-Bolt
!pip install "timesfm[torch]"             # TimesFM 2.5
!pip install uni2ts                       # Moirai
!pip install tabpfn-time-series           # TabPFN-TS (cloud client = no GPU)
!pip install "autogluon.timeseries>=1.5"

from __future__ import annotations
import os, json, math, gc, warnings
from pathlib import Path
from typing import List, Dict, Callable, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ===========================================================================
# CONFIG  — edit these
# ===========================================================================
CONFIG = {
    # --- paths --------------------------------------------------------------
    "cache_dir":     os.path.join(root, "cache","44"),  # has feature_stats.json + series/
    "tft_forecasts": os.path.join(root, "tft_epd_forecast_44","forecasts.csv"),                          # the TFT's output (defines windows)
    "out_dir":       os.path.join(root,"fm_bench_out_44"),
    "tft_name":      "ModernTFT",   # label for the TFT column in the comparison
    "chronos_model": "autogluon/chronos-bolt-base",   # or "autogluon/chronos-2" for covariates + better tails

    # --- evaluation knobs ---------------------------------------------------
    # Quantiles we ask every model for. p50 is the point forecast; the rest feed
    # pinball / coverage. (TimesFM only emits deciles up to 0.9 — 0.97 is filled
    # by its top decile, flagged in code.)
    "quantile_levels": [0.1, 0.25, 0.5, 0.75, 0.9],
    "context_mode": "encoder_len",   # "encoder_len" (fair) | "all" (full history)
    "min_context":  12,              # skip windows with fewer than this many months
    "seasonality":  12,              # monthly -> 12, used for MASE + seasonal-naive
    "batch_size":   256,             # windows per model call (tune to your GPU)
    # Auto-detect: GPU FMs (Moirai/TimesFM) silently failed on a CPU Colab run
    # because this was hardcoded to "cuda". Override to "cpu"/"cuda" if needed.
    "device":       ("cuda" if __import__("torch").cuda.is_available() else "cpu"),

    # --- which models to run ------------------------------------------------
    "models": {
        "SeasonalNaive": True,    # free, no deps — always run this baseline
        "Chronos":       True,    # pip install chronos-forecasting
        "TimesFM":       True,    # pip install timesfm[torch]
        "Moirai":        True,    # pip install uni2ts
        "TabPFN":        True,    # pip install tabpfn-time-series
    },

    # --- model checkpoints --------------------------------------------------
    "chronos_model": "amazon/chronos-bolt-base",        # or chronos-bolt-small / amazon/chronos-2
    "timesfm_model": "google/timesfm-2.5-200m-pytorch",
    "moirai_model":  "Salesforce/moirai-1.1-R-large",   # or moirai-2.0-R-small
    "moirai_patch":  "auto",
    "tabpfn_mode":   "client",   # "client" (cloud, no GPU) | "local" (needs CUDA)
}

QCOLS = lambda qs: [f"p{int(round(q*100))}" for q in qs]


# ===========================================================================
# PART 1 — Data harness: rebuild item-unit series + the TFT's test windows
# ===========================================================================
class Windows:
    """Holds every (series, origin) task: context history + horizon target."""
    def __init__(self, tasks, stats, qlevels):
        self.tasks = tasks          # list of dicts (see _build)
        self.stats = stats
        self.qlevels = qlevels

    def __len__(self): return len(self.tasks)


def load_harness(cfg) -> Windows:
    cache = Path(cfg["cache_dir"])
    stats = json.load(open(cache / "feature_stats.json"))
    L, H = int(stats["encoder_len"]), int(stats["horizon"])
    t_mean, t_std = float(stats["t_mean"]), float(stats["t_std"])
    base_ord = int(stats["base_year"]) * 12 + int(stats["base_moy"])  # global idx 0
    n_months = int(stats["n_months"])

    # invert standardized log1p(items) channel 0 -> raw item counts, per series
    def items_of(sid: int) -> np.ndarray:
        z = np.load(cache / "series" / f"{sid}.npy")          # (n_months, 4) float32
        logdemand = z[:, 0].astype(np.float64) * t_std + t_mean
        return np.expm1(logdemand)                            # items, length n_months

    # reconstruct the evaluation origins straight from the TFT's forecasts.csv
    fc = pd.read_csv(cfg["tft_forecasts"])
    yr = fc["year_month"].str.slice(0, 4).astype(int)
    mo = fc["year_month"].str.slice(5, 7).astype(int)
    fc["gidx"]   = yr * 12 + (mo - 1) - base_ord              # month being forecast
    fc["origin"] = fc["gidx"] - (fc["step"] - 1)             # 1st horizon month idx

    item_cache: Dict[int, np.ndarray] = {}
    tasks = []
    skipped = 0
    for (sid, origin), g in fc.groupby(["series_id", "origin"]):
        sid = int(sid); origin = int(origin)
        if sid not in item_cache:
            item_cache[sid] = items_of(sid)
        series = item_cache[sid]

        ctx_start = (origin - L) if cfg["context_mode"] == "encoder_len" else 0
        ctx_start = max(0, ctx_start)
        context = series[ctx_start:origin]
        context = context[np.isfinite(context)]
        if len(context) < cfg["min_context"]:
            skipped += 1
            continue

        g = g.sort_values("step")
        tasks.append(dict(
            series_id=sid, origin=origin,
            context=context.astype(np.float64),
            target=g["target"].to_numpy(np.float64),          # (h,)
            year_month=g["year_month"].tolist(),
            steps=g["step"].to_numpy(int),
        ))

    print(f"[harness] {len(tasks)} windows  (skipped {skipped} for short context)  "
          f"L={L} H={H}  context_mode={cfg['context_mode']}")
    stats["_L"], stats["_H"] = L, H
    return Windows(tasks, stats, cfg["quantile_levels"])


# ===========================================================================
# PART 2 — Model adapters
# Each adapter exposes .predict(contexts, horizon, qlevels) -> (N, H, Q) array.
# contexts: list of 1-D float arrays (variable length). Returns item-unit quantiles.
# ===========================================================================
def _batched(seq, bs):
    for i in range(0, len(seq), bs):
        yield seq[i:i + bs]


def _resolve_qcol(df_columns, q):
    """Find the column in `df_columns` that holds quantile level `q`.

    Different libraries name quantile columns differently — TabPFN-TS and
    AutoGluon have used "0.1", "0.10", 0.1 (float), "q0.1", and "quantile_0.1"
    across versions, and the median is sometimes only available as "mean"/
    "target". This resolver tries every spelling and matches floats with a
    tolerance, so a naming change no longer crashes the whole benchmark with a
    KeyError (the original TabPFN failure mode)."""
    cols = list(df_columns)
    # 1) exact string spellings
    candidates = [str(q), f"{q:.1f}", f"{q:.2f}", f"q{q}", f"q{q:.1f}",
                  f"quantile_{q}", f"quantile_{q:.1f}", f"{int(round(q*100))}",
                  f"p{int(round(q*100))}"]
    for cand in candidates:
        if cand in cols:
            return cand
    # 2) numeric match (column header is itself a float/near-float)
    for c in cols:
        try:
            if abs(float(c) - q) < 1e-6:
                return c
        except (TypeError, ValueError):
            continue
    # 3) median fallbacks
    if abs(q - 0.5) < 1e-6:
        for cand in ("mean", "target", "0.5", "median"):
            if cand in cols:
                return cand
    return None


class SeasonalNaiveAdapter:
    """y_hat[h] = last value from the same season (t - season). Quantiles come
    from the empirical residual distribution of the in-context seasonal-naive
    errors — a genuine, defensible probabilistic baseline (not a toy)."""
    name = "SeasonalNaive"
    def __init__(self, cfg, season): self.season = season
    def load(self): pass
    def predict(self, contexts, horizon, qlevels):
        out = np.zeros((len(contexts), horizon, len(qlevels)))
        z = {q: 0.0 for q in qlevels}
        for i, c in enumerate(contexts):
            # FIX: use the seasonal period whenever we have AT LEAST one full
            # season of context (>=, not >). With encoder_len==season==12 the
            # old `>` silently fell back to s=1 (persistence), so the "seasonal
            # naive" baseline meant different things at encoder_len 12 vs 24.
            # `>=` makes it a TRUE seasonal-naive at any encoder_len.
            s = self.season if len(c) >= self.season else 1
            # point: repeat the last full season across the horizon
            base = np.array([c[-s + (h % s)] for h in range(horizon)])
            # residual scale from in-sample seasonal-naive errors. Keep `>` here:
            # we need strictly MORE than one season to form any (c[s:]-c[:-s])
            # differences; with exactly one season we fall back to a 5%-of-mean sd.
            if len(c) > s:
                res = c[s:] - c[:-s]
                sd = np.std(res) if np.std(res) > 1e-6 else max(1.0, 0.05 * np.mean(c))
            else:
                sd = max(1.0, 0.05 * np.mean(c))
            for j, q in enumerate(qlevels):
                from scipy.stats import norm
                out[i, :, j] = np.maximum(0.0, base + norm.ppf(q) * sd * np.sqrt(np.arange(1, horizon + 1)))
        return out



class ChronosAdapter:
    """Chronos via AutoGluon-TimeSeries (the HF-card route).
    INSTALL: pip install "autogluon.timeseries>=1.5"
    Each (series, origin) context window is one item_id; .fit() does NOT train
    (zero-shot) — it just primes the predictor; .predict() returns the next H steps."""
    name = "Chronos"
    def __init__(self, cfg): self.cfg = cfg
    def load(self):
        from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame
        self.TSP, self.TSDF = TimeSeriesPredictor, TimeSeriesDataFrame
    def predict(self, contexts, horizon, qlevels):
        import pandas as pd, numpy as np
        # long frame: one item per window, dummy monthly index (Chronos is date-agnostic)
        frames = []
        for i, c in enumerate(contexts):
            ts = pd.date_range("2000-01-01", periods=len(c), freq="MS")
            frames.append(pd.DataFrame({"item_id": i, "timestamp": ts,
                                        "target": np.asarray(c, dtype="float64")}))
        df = self.TSDF.from_data_frame(pd.concat(frames, ignore_index=True))
        predictor = self.TSP(
            prediction_length=horizon,
            quantile_levels=list(qlevels),       # e.g. [0.1,0.25,0.5,0.75,0.9,0.97]
            freq="MS", verbosity=0,
        ).fit(
            df,
            hyperparameters={"Chronos": {"model_path": self.cfg["chronos_model"]}},
            enable_ensemble=False,               # Chronos only, no ensemble/training
            skip_model_selection=True,
        )
        pred = predictor.predict(df).reset_index()  # cols: item_id, timestamp, mean, "<q>"...
        out = np.zeros((len(contexts), horizon, len(qlevels)))
        for i, g in pred.groupby("item_id"):
            g = g.sort_values("timestamp")
            for j, q in enumerate(qlevels):
                col = _resolve_qcol(g.columns, q)
                if col is None:
                    raise KeyError(
                        f"Chronos: no column for quantile {q}; got {list(g.columns)}")
                out[int(i), :, j] = g[col].to_numpy()[:horizon]
        return out


class TimesFMAdapter:
    """TimesFM 2.5. INSTALL: pip install timesfm[torch]
    forecast() returns quantile_forecast (B,H,10): [mean, q10..q90]. We map the
    requested levels onto those deciles by linear interpolation; anything above
    0.9 (e.g. 0.97) is clamped to the top decile and FLAGGED in the comparison."""
    name = "TimesFM"
    def __init__(self, cfg): self.cfg = cfg
    def load(self):
        import torch, timesfm
        self.np = np
        self.model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(self.cfg["timesfm_model"])
        self.model.compile(timesfm.ForecastConfig(
            max_context=1024, max_horizon=max(64, 8),
            normalize_inputs=True, use_continuous_quantile_head=True,
            force_flip_invariance=True, infer_is_positive=True,
            fix_quantile_crossing=True,
        ))
        self.decile_q = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
    def predict(self, contexts, horizon, qlevels):
        out = []
        for chunk in _batched(contexts, self.cfg["batch_size"]):
            _pt, qf = self.model.forecast(horizon=horizon, inputs=[c for c in chunk])
            qf = np.asarray(qf)                       # (B,H,10): idx0=mean, 1..9=q10..q90
            deciles = qf[:, :, 1:10]                  # (B,H,9)
            mapped = np.empty((qf.shape[0], horizon, len(qlevels)))
            for j, q in enumerate(qlevels):
                qq = min(q, 0.9)                      # clamp >0.9 to top decile
                mapped[:, :, j] = self._interp(deciles, qq)
            out.append(mapped)
        return np.concatenate(out, 0)
    def _interp(self, deciles, q):
        # linear interp across the 9 decile levels (0.1..0.9)
        idx = (q - 0.1) / 0.1
        lo = int(np.floor(idx)); hi = min(lo + 1, 8); w = idx - lo
        return deciles[:, :, lo] * (1 - w) + deciles[:, :, hi] * w


class MoiraiAdapter:
    """Moirai (uni2ts). INSTALL: pip install uni2ts
    Uses GluonTS ListDataset + the Moirai predictor; SampleForecast.quantile(q)."""
    name = "Moirai"
    def __init__(self, cfg): self.cfg = cfg
    def load(self):
        import torch
        from gluonts.dataset.common import ListDataset
        self.ListDataset = ListDataset
        self.torch = torch
        mpath = self.cfg["moirai_model"]
        if "moirai-2" in mpath:
            from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module
            self.Forecast, self.Module = Moirai2Forecast, Moirai2Module
        else:
            from uni2ts.model.moirai import MoiraiForecast, MoiraiModule
            self.Forecast, self.Module = MoiraiForecast, MoiraiModule
        self.module = self.Module.from_pretrained(mpath)
    def _predictor(self, ctx_len, horizon):
        psz = self.cfg["moirai_patch"]
        model = self.Forecast(
            module=self.module, prediction_length=horizon,
            context_length=ctx_len, patch_size=psz,
            num_samples=200, target_dim=1,
            feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0,
        )
        return model.create_predictor(batch_size=self.cfg["batch_size"],
                                      device=self.cfg["device"])
    def predict(self, contexts, horizon, qlevels):
        # group by context length so one predictor serves a homogeneous batch
        out = [None] * len(contexts)
        by_len: Dict[int, List[int]] = {}
        for i, c in enumerate(contexts): by_len.setdefault(len(c), []).append(i)
        for ctx_len, idxs in by_len.items():
            entries = [{"start": pd.Period("2000-01", "M"),
                        "target": contexts[i].astype(np.float32)} for i in idxs]
            ds = self.ListDataset(entries, freq="M")
            predictor = self._predictor(ctx_len, horizon)
            for k, fc in enumerate(predictor.predict(ds)):
                arr = np.stack([fc.quantile(q) for q in qlevels], axis=-1)  # (H,Q)
                out[idxs[k]] = arr
        return np.stack(out, 0)


class TabPFNAdapter:
    """TabPFN-TS. INSTALL: pip install tabpfn-time-series
    Cloud client by default (no GPU). Frames each window as tabular regression."""
    name = "TabPFN"
    def __init__(self, cfg): self.cfg = cfg
    def load(self):
        from tabpfn_time_series import TabPFNTSPipeline
        self.pipeline = TabPFNTSPipeline(
            tabpfn_mode=self.cfg.get("tabpfn_mode", "client"),
        )
    def predict(self, contexts, horizon, qlevels):
        # Build one long context_df (item_id, timestamp, target); predict_df returns
        # quantile columns named by level. We request our qlevels where supported.
        frames = []
        for i, c in enumerate(contexts):
            ts = pd.period_range("2000-01", periods=len(c), freq="M").to_timestamp()
            frames.append(pd.DataFrame({"item_id": i, "timestamp": ts, "target": c}))
        ctx_df = pd.concat(frames, ignore_index=True)
        pred = self.pipeline.predict_df(
            ctx_df, prediction_length=horizon, quantiles=list(qlevels),
        )                                          # long df: item_id,timestamp,<qcols>
        out = np.zeros((len(contexts), horizon, len(qlevels)))
        for i, (_iid, g) in enumerate(pred.groupby("item_id")):
            g = g.sort_values("timestamp")
            for j, q in enumerate(qlevels):
                col = _resolve_qcol(g.columns, q)
                if col is None:
                    raise KeyError(
                        f"TabPFN: no column for quantile {q}; got {list(g.columns)}")
                out[i, :, j] = g[col].to_numpy()[:horizon]
        return out


ADAPTERS = {
    "SeasonalNaive": lambda cfg: SeasonalNaiveAdapter(cfg, cfg["seasonality"]),
    "Chronos":       ChronosAdapter,
    "TimesFM":       TimesFMAdapter,
    "Moirai":        MoiraiAdapter,
    "TabPFN":        TabPFNAdapter,
}


# ===========================================================================
# PART 3 — Runner: run one model over all windows, write its forecasts CSV
# ===========================================================================
def run_model(name, adapter, win: Windows, cfg) -> str:
    H, qlevels = win.stats["_H"], win.qlevels
    contexts = [t["context"] for t in win.tasks]
    print(f"\n[{name}] forecasting {len(contexts)} windows ...")
    adapter.load()
    preds = adapter.predict(contexts, H, qlevels)        # (N, H, Q) item units
    preds = np.clip(preds, 0, None)
    # enforce monotone quantiles (guards against crossing)
    preds = np.sort(preds, axis=-1)

    qcols = QCOLS(qlevels)
    rows = []
    for n, t in enumerate(win.tasks):
        sid = t["series_id"]
        for h in range(len(t["steps"])):
            ym = t["year_month"][h]
            rows.append([f"{sid}_{ym}", sid, ym, int(t["steps"][h]),
                         *preds[n, h].tolist(), float(t["target"][h])])
    df = pd.DataFrame(rows, columns=["row_id", "series_id", "year_month", "step",
                                     *qcols, "target"])
    Path(cfg["out_dir"]).mkdir(parents=True, exist_ok=True)
    out = os.path.join(cfg["out_dir"], f"forecasts_{name}.csv")
    df.to_csv(out, index=False)
    print(f"[{name}] wrote {out}  ({len(df):,} rows)")
    del adapter, preds; gc.collect()
    return out


# ===========================================================================
# PART 4 — Comparison: every model CSV (+ the TFT's) vs the target
# ===========================================================================
def _pinball(target, pred, q):
    e = target - pred
    return np.mean(np.maximum(q * e, (q - 1) * e))

def _mase_scale(win: Windows, season):
    # per-window in-sample seasonal-naive MAE, averaged — the MASE denominator
    scales = []
    for t in win.tasks:
        c = t["context"]; s = season if len(c) > season else 1
        scales.append(np.mean(np.abs(c[s:] - c[:-s])) if len(c) > s else np.mean(np.abs(np.diff(c))))
    return float(np.mean([x for x in scales if np.isfinite(x) and x > 0]))

def evaluate_csv(path, mase_scale, season):
    df = pd.read_csv(path)
    a, f = df["target"].to_numpy(), df["p50"].to_numpy()
    e = a - f
    res = {
        "n_rows": len(df),
        "WAPE":  np.sum(np.abs(e)) / np.sum(np.abs(a)),
        "MAE":   np.mean(np.abs(e)),
        "RMSE":  np.sqrt(np.mean(e ** 2)),
        "bias":  np.mean(f - a),
        "MASE":  np.mean(np.abs(e)) / mase_scale,
        "sMAPE": np.mean(2 * np.abs(e) / (np.abs(a) + np.abs(f) + 1e-8)),
    }
    # probabilistic, over whatever quantile columns exist
    qcols = [c for c in df.columns if c.startswith("p") and c[1:].isdigit()]
    qs = [int(c[1:]) / 100 for c in qcols]
    if qcols:
        pin = np.mean([_pinball(a, df[c].to_numpy(), q) for c, q in zip(qcols, qs)])
        res["pinball"] = pin
        if "p25" in df.columns and "p75" in df.columns:   # 50% IQR
            res["cov_25_75"] = np.mean((a >= df["p25"]) & (a <= df["p75"]))
        if "p10" in df.columns and "p90" in df.columns:   # 80% interval
            res["cov_10_90"] = np.mean((a >= df["p10"]) & (a <= df["p90"]))
    # per-horizon WAPE
    ph = {}
    for s, g in df.groupby("step"):
        ee = g["target"].to_numpy() - g["p50"].to_numpy()
        ph[int(s)] = np.sum(np.abs(ee)) / np.sum(np.abs(g["target"].to_numpy()))
    return res, ph

def _wape(a, f):
    return np.sum(np.abs(a - f)) / (np.sum(np.abs(a)) + 1e-12)


def paired_significance(model_csvs: Dict[str, str], cfg, reference=None,
                        n_boot=2000, seed=0):
    """Is the reference model (the TFT) *significantly* better than each rival?

    WHY THIS IS NEEDED
    ------------------
    The headline WAPE is a single number over ~10k rows, but those rows are NOT
    independent: they are 144 series x rolling origins x 6 horizon steps. The
    effective sample size is closer to the number of SERIES. A 0.043 vs 0.044
    WAPE gap is meaningless unless we account for that clustering. We do two
    standard things:

      1) CLUSTER (BLOCK) BOOTSTRAP over series. We resample whole series with
         replacement n_boot times; for each draw we recompute every model's WAPE
         on the resampled cohort and record (rival_WAPE - reference_WAPE). The
         2.5/97.5 percentiles give a 95% CI on the gap; the share of draws where
         the reference wins is a one-sided bootstrap p-value. A CI that straddles
         0 means "statistical tie" — which is the honest verdict for TFT vs the
         best foundation model here.

      2) DIEBOLD-MARIANO test on the per-row absolute-error differential
         d = |e_rival| - |e_reference|, with a cluster-robust SE (clustered by
         series). DM > 0 with small p => reference is more accurate.

    Output: significance_vs_<reference>.csv  (one row per rival).
    """
    rng = np.random.default_rng(seed)
    reference = reference or cfg.get("tft_name", "ModernTFT")

    # load aligned (row_id-keyed) p50 errors per model that exists
    loaded = {}
    for name, path in model_csvs.items():
        if path and os.path.exists(path):
            d = pd.read_csv(path)[["row_id", "series_id", "target", "p50"]].copy()
            loaded[name] = d.set_index("row_id")
    if reference not in loaded:
        print(f"[significance] reference '{reference}' missing — skipping tests.")
        return None

    ref = loaded[reference]
    rows = []
    for name, d in loaded.items():
        if name == reference:
            continue
        # align rival to reference on row_id (inner join keeps shared windows only)
        j = ref.join(d[["p50"]], rsuffix="_riv", how="inner")
        a = j["target"].to_numpy(float)
        f_ref = j["p50"].to_numpy(float)
        f_riv = j["p50_riv"].to_numpy(float)
        sid = j["series_id"].to_numpy()
        ae_ref = np.abs(a - f_ref)
        ae_riv = np.abs(a - f_riv)

        # point gap in WAPE (rival - reference); positive => reference better
        gap = _wape(a, f_riv) - _wape(a, f_ref)

        # ---- (1) cluster bootstrap over series ----
        uniq = np.unique(sid)
        by_sid = {s: np.where(sid == s)[0] for s in uniq}
        boot = np.empty(n_boot)
        for b in range(n_boot):
            pick = rng.choice(uniq, size=len(uniq), replace=True)
            idx = np.concatenate([by_sid[s] for s in pick])
            boot[b] = _wape(a[idx], f_riv[idx]) - _wape(a[idx], f_ref[idx])
        lo, hi = np.percentile(boot, [2.5, 97.5])
        p_ref_wins = float(np.mean(boot > 0))      # one-sided: reference better

        # ---- (2) Diebold-Mariano with series-clustered SE ----
        d_loss = ae_riv - ae_ref                    # >0 => reference better
        dbar = float(np.mean(d_loss))
        # cluster-robust variance of the mean (clustered by series)
        clus = np.array([d_loss[by_sid[s]].sum() for s in uniq])
        n = len(d_loss); G = len(uniq)
        var_mean = (np.sum((clus - n * dbar / G) ** 2)) / (n ** 2) * (G / max(1, G - 1))
        se = math.sqrt(var_mean) if var_mean > 0 else float("nan")
        dm = dbar / se if se and np.isfinite(se) and se > 0 else float("nan")
        # two-sided normal-approx p-value
        from math import erfc, sqrt as _sqrt
        dm_p = float(erfc(abs(dm) / _sqrt(2))) if np.isfinite(dm) else float("nan")

        verdict = ("reference SIGNIFICANTLY better" if lo > 0 else
                   "rival SIGNIFICANTLY better" if hi < 0 else
                   "TIE (CI straddles 0)")
        rows.append({
            "rival": name, "reference": reference,
            "WAPE_gap(rival-ref)": gap,
            "boot_CI95_lo": lo, "boot_CI95_hi": hi,
            "p_reference_wins": p_ref_wins,
            "DM_stat": dm, "DM_p_value": dm_p,
            "n_shared_rows": int(len(a)), "n_series": int(G),
            "verdict": verdict,
        })

    sig = pd.DataFrame(rows)
    Path(cfg["out_dir"]).mkdir(parents=True, exist_ok=True)
    sig.to_csv(os.path.join(cfg["out_dir"], f"significance_vs_{reference}.csv"), index=False)
    pd.set_option("display.width", 220, "display.max_columns", 30)
    print(f"\n================  SIGNIFICANCE: {reference} vs each rival  ================")
    print("(positive gap / CI above 0 => the TFT is genuinely better; "
          "CI straddling 0 => statistical tie)")
    print(sig.round(4).to_string(index=False))
    return sig


def load_train_timing(cfg):
    """Read the TFT training-time record written by EPDForecastTrainer.train()."""
    candidates = [
        os.path.join(os.path.dirname(cfg["tft_forecasts"]), "logs", "train_timing.json"),
        os.path.join(root, "tft_epd_forecast_2", "logs", "train_timing.json"),
    ]
    for p in candidates:
        if os.path.exists(p):
            with open(p) as f:
                return json.load(f), p
    return None, None


def compare(model_csvs: Dict[str, str], win: Windows, cfg):
    season = cfg["seasonality"]
    scale = _mase_scale(win, season)
    head, perh = {}, {}
    for name, path in model_csvs.items():
        if not (path and os.path.exists(path)):
            continue
        r, ph = evaluate_csv(path, scale, season)
        head[name] = r; perh[name] = ph

    summary = pd.DataFrame(head).T
    order = ["WAPE", "MASE", "MAE", "RMSE", "sMAPE", "bias",
             "pinball", "cov_25_75", "cov_10_90", "n_rows"]
    summary = summary[[c for c in order if c in summary.columns]]
    summary = summary.sort_values("WAPE")
    perh_df = pd.DataFrame(perh).T
    perh_df.columns = [f"h{c}" for c in perh_df.columns]

    # ---- training-cost column: TFT pays a one-time cost; FMs are zero-shot ----
    timing, tpath = load_train_timing(cfg)
    if timing is not None:
        tft_label = cfg.get("tft_name", "ModernTFT")
        summary["train_minutes"] = 0.0          # foundation models: zero-shot
        if tft_label in summary.index:
            summary.loc[tft_label, "train_minutes"] = round(timing["total_train_minutes"], 2)
        print(f"\n[cost] TFT trained in {timing['total_train_minutes']:.2f} min "
              f"on {timing['device']} ({timing['epochs_run']} epochs, "
              f"{timing['n_params']:,} params). FMs are zero-shot (0 min). "
              f"[{tpath}]")

    Path(cfg["out_dir"]).mkdir(parents=True, exist_ok=True)
    summary.to_csv(os.path.join(cfg["out_dir"], "comparison_summary.csv"))
    perh_df.to_csv(os.path.join(cfg["out_dir"], "comparison_per_horizon.csv"))

    pd.set_option("display.width", 200, "display.max_columns", 30)
    print("\n================  HEADLINE (sorted by WAPE)  ================")
    print(summary.round(4).to_string())
    print("\n================  WAPE per horizon  ================")
    print(perh_df.round(4).to_string())
    print(f"\n(MASE denominator = seasonal-naive in-sample MAE = {scale:.2f} items)")

    return summary, perh_df


# ===========================================================================
# Main
# ===========================================================================
def main(cfg=CONFIG):
    win = load_harness(cfg)
    model_csvs: Dict[str, Optional[str]] = {}

    # include the TFT itself in the comparison
    if os.path.exists(cfg["tft_forecasts"]):
        model_csvs[cfg["tft_name"]] = cfg["tft_forecasts"]

    for name, on in cfg["models"].items():
        if not on:
            continue
        try:
            adapter = ADAPTERS[name](cfg)
            model_csvs[name] = run_model(name, adapter, win, cfg)
        except Exception as ex:
            print(f"[{name}] SKIPPED — {type(ex).__name__}: {ex}")
            model_csvs[name] = None

    compare(model_csvs, win, cfg)
    # cluster-bootstrap + Diebold-Mariano significance of TFT vs every rival
    paired_significance(model_csvs, cfg, reference=cfg.get("tft_name", "ModernTFT"))


if __name__ == "__main__":
    main()

⠏ Authenticating

Login successful!

Predicting time series: 100%|██████████| 1721/1721 [08:05<00:00,  3.54it/s]


[TabPFN] wrote /content/drive/MyDrive/Dr. Moustafa Saad/mine/TFT/fm_bench_out_44/forecasts_TabPFN.csv  (10,326 rows)

[cost] TFT trained in 11.66 min on cpu (42 epochs, 627,919 params). FMs are zero-shot (0 min). [/content/drive/MyDrive/Dr. Moustafa Saad/mine/TFT/tft_epd_forecast_44/logs/train_timing.json]

================  HEADLINE (sorted by WAPE)  ================
                 WAPE    MASE         MAE         RMSE   sMAPE        bias     pinball  cov_25_75  cov_10_90   n_rows  train_minutes
SeasonalNaive  0.0359  0.6635  26503.0133   58438.3176  0.0783 -15846.4961  11173.2914     0.7408     0.9185  10326.0           0.00
TabPFN         0.0432  0.7971  31840.2623   76254.5750  0.0920 -12734.9935  11498.5945     0.3833     0.6836  10326.0           0.00
TimesFM        0.0440  0.8131  32478.8842   78891.1325  0.1052 -12309.1217  11723.3163     0.4663     0.8028  10326.0           0.00
ModernTFT      0.0444  0.8202  32762.2987   65258.4604  0.0935   4592.4049  12145.9818     0.4386